# AI Programming 코딩 시험 대비 실습 노트북
**I060-3-4024-01 · Fall 2026** — 짝 문서: `AI_Programming_Coding_Exam_Codebook.docx` (문제 ID 동일)

**시험 조건 (Lecture11_Lab2 p.3):** Coding Test 10/21(수) 12:00–13:15 · Google Colab 전용 · 인터넷 검색·강의자료 허용 · 생성형 AI 금지(“Hide generative AI features” 켜기)

## 사용법
1. **[0] 공통 SETUP 셀을 먼저 1회 실행** → 이후 모든 정답 셀은 서로 독립 실행됩니다 (한 셀이 에러나도 다른 문제 연습 가능).
2. 각 섹션은 **문제(연습 셀) → 정답 섹션** 순서. 연습 셀의 `____`(빈칸)·`TODO`를 채워 실행, 정답은 섹션 끝 “정답” 부분에서 확인.
3. CNN 문제는 **런타임 > 런타임 유형 변경 > GPU** 권장. 시간이 부족하면 `EPOCHS`를 줄이세요.
4. `Ctrl+F`(Colab: `Ctrl+H` 찾기/바꾸기)로 문제 ID(예: `E04`)나 API 이름(예: `ImageDataGenerator`)을 검색.

## 데이터 출처
| 데이터 | 우선 경로 | 없을 때 대체 (자동) |
|---|---|---|
| Pima / Iris / Wine | 강의 CSV (`DATA_DIR` = Lab1 경로) | OpenML diabetes / sklearn iris / UCI Wine Quality red+white |
| Ames Housing | `ames_train.csv` (Lab2) | OpenML `house_prices` (Kaggle 원본) |
| MSFT | 시험에서 주어진 CSV (`MSFT_CSV` 수정) | `yfinance` 실제 MSFT → 불가 시 **SYNTHETIC(가짜, 확인용)** 명시 |
| ETTh1 / IRB2400 | 강의 CSV | ETDataset GitHub 원본 / sklearn linnerud(다른 데이터, 구조 연습용) |
| Fashion-MNIST | `tensorflow.keras.datasets.fashion_mnist` (자동 다운로드) | — |

대체 데이터는 실행 시 `[data]` 메시지로 무엇을 썼는지 표시됩니다. 원본 데이터인 척하지 않습니다.

## 버전 메모 (Colab 기본 TF 2.16+ / Keras 3 기준)
- 모델 저장 확장자 `.keras` 필수 (`ModelCheckpoint`, `model.save`) · `LeakyReLU(negative_slope=0.1)` (구버전 `alpha`)
- `ImageDataGenerator`는 deprecated지만 동작함(강의 방식). 대안: E12 `RandomFlip` 등 · generator 입력에 `validation_split` 불가
- `OneHotEncoder(sparse_output=False)` (sklearn ≥ 1.2) · 사용자 정의 손실에서 `tf.cast(y_true, y_pred.dtype)`

**출제 근거 표기:** `족보 직접 등장` = 사용자 제공 족보 목록 항목 / `강의자료 기반 예상` = 첨부 강의자료 코드 근거 예측 / `일반 구현` = 해당 강의자료 미첨부·강의 외 대체 구현.

## [0] 공통 SETUP

In [ ]:
# ===== [0] 공통 SETUP — 노트북을 열면 이 셀을 가장 먼저 1회 실행 =====
# 이후 모든 정답 셀은 이 셀만 실행되어 있으면 서로 독립적으로 실행된다.
# Colab 기본 설치: numpy, pandas, matplotlib, seaborn, scikit-learn, tensorflow(Keras 3)
# MSFT 자동 다운로드가 필요할 때만:  !pip install -q yfinance
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ↓↓↓ 강의자료(Lab1/Lab2)와 같은 Drive 경로. 본인 경로로 수정 ↓↓↓
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/data'
MODEL_DIR = '/content/drive/MyDrive/Colab Notebooks/models'

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']   # Fashion-MNIST 0~9


def mount_drive():
    """Colab이면 Google Drive 마운트 (Lab1 p.5)."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('[info] Drive 마운트 생략:', type(e).__name__)


def _csv(name):
    return os.path.join(DATA_DIR, name)


# ---------- Lab1: Pima (pima-indians-diabetes3.csv) ----------
def load_pima():
    path = _csv('pima-indians-diabetes3.csv')
    if os.path.exists(path):
        print('[data] 강의 CSV:', path)
        return pd.read_csv(path)
    print('[data] 강의 CSV 없음 → 대체: OpenML "diabetes"(원본 Pima 데이터). 열 이름만 강의자료 형식으로 변경')
    from sklearn.datasets import fetch_openml
    d = fetch_openml(name='diabetes', version=1, as_frame=True).frame
    d.columns = ['pregnant', 'plasma', 'pressure', 'thickness', 'insulin',
                 'bmi', 'pedigree', 'age', 'diabetes']
    d['diabetes'] = (d['diabetes'].astype(str) == 'tested_positive').astype(int)
    return d


# ---------- Lab1: Iris (iris3.csv) ----------
def load_iris_df():
    path = _csv('iris3.csv')
    if os.path.exists(path):
        print('[data] 강의 CSV:', path)
        return pd.read_csv(path)
    print('[data] 강의 CSV 없음 → 대체: sklearn load_iris (동일한 Fisher Iris 150개). 열 이름은 대체 데이터 기준')
    from sklearn.datasets import load_iris
    iris = load_iris()
    d = pd.DataFrame(iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    d['species'] = ['Iris-' + iris.target_names[t] for t in iris.target]
    return d


# ---------- Lab1: Wine red/white (wine.csv, header=None) ----------
def load_wine_rw():
    path = _csv('wine.csv')
    if os.path.exists(path):
        print('[data] 강의 CSV:', path)
        return pd.read_csv(path, header=None)
    try:
        base = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/'
        red = pd.read_csv(base + 'winequality-red.csv', sep=';')
        white = pd.read_csv(base + 'winequality-white.csv', sep=';')
        red['label'], white['label'] = 1, 0
        d = pd.concat([red, white], ignore_index=True)
        d.columns = range(d.shape[1])            # 강의 CSV처럼 0~12 정수 열
        print('[data] 강의 CSV 없음 → 대체: UCI Wine Quality red+white (1: red, 0: white)')
        return d
    except Exception as e:
        print('[data] wine 대체 다운로드 실패:', type(e).__name__, '→ wine.csv를 DATA_DIR에 업로드하세요')
        raise


# ---------- Lab2: Ames Housing (ames_train.csv) ----------
def load_ames():
    path = _csv('ames_train.csv')
    if os.path.exists(path):
        print('[data] 강의 CSV:', path)
        return pd.read_csv(path)
    print('[data] 강의 CSV 없음 → 대체: OpenML "house_prices"(Kaggle House Prices train, 1460행)')
    from sklearn.datasets import fetch_openml
    d = fetch_openml(name='house_prices', as_frame=True).frame
    d = d.apply(lambda s: s.astype(object) if s.dtype.name == 'category' else s)   # fillna('None') 가능하게
    if 'Id' not in d.columns:
        d.insert(0, 'Id', np.arange(1, len(d) + 1))
    return d


ABSENCE_CAT_COLS = ['PoolQC', 'Alley', 'Fence', 'FireplaceQu', 'MiscFeature',
                    'MasVnrType', 'GarageType', 'GarageFinish', 'GarageQual',
                    'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
                    'BsmtFinType1', 'BsmtFinType2']
ABSENCE_NUM_COLS = ['GarageYrBlt', 'MasVnrArea']


def prepare_ames(df, seed=42):
    """Lab2 p.8~11 전처리 파이프라인 그대로 (split → fillna → OneHot → StandardScaler → concatenate)."""
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    X = df.drop(columns=['Id', 'SalePrice'], errors='ignore').copy()
    y = df['SalePrice'].astype(float).copy()
    X['MSSubClass'] = X['MSSubClass'].astype(str)
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=seed)
    X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()
    for data in [X_train, X_val, X_test]:
        data[ABSENCE_CAT_COLS] = data[ABSENCE_CAT_COLS].fillna('None')
        data[ABSENCE_NUM_COLS] = data[ABSENCE_NUM_COLS].fillna(0)
    lot_median = X_train['LotFrontage'].median()          # train 기준 (누수 방지)
    elec_mode = X_train['Electrical'].mode()[0]
    for data in [X_train, X_val, X_test]:
        data['LotFrontage'] = data['LotFrontage'].fillna(lot_median)
        data['Electrical'] = data['Electrical'].fillna(elec_mode)
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
    num_cols = X_train.select_dtypes(include='number').columns
    for data in [X_train, X_val, X_test]:                 # 남은 결측(대체 데이터 대비 안전장치)
        data[cat_cols] = data[cat_cols].astype(str).replace('nan', 'None')
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_cat = encoder.fit_transform(X_train[cat_cols])
    X_val_cat, X_test_cat = encoder.transform(X_val[cat_cols]), encoder.transform(X_test[cat_cols])
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[num_cols].fillna(0))
    X_val_num = scaler.transform(X_val[num_cols].fillna(0))
    X_test_num = scaler.transform(X_test[num_cols].fillna(0))
    X_train_input = np.concatenate([X_train_num, X_train_cat], axis=1)
    X_val_input = np.concatenate([X_val_num, X_val_cat], axis=1)
    X_test_input = np.concatenate([X_test_num, X_test_cat], axis=1)
    return X_train_input, X_val_input, X_test_input, y_train, y_val, y_test


# ---------- 족보: MSFT 주가 (시험에서 주어진 CSV 파일명으로 수정) ----------
MSFT_CSV = _csv('MSFT.csv')     # ← 예시 경로. 시험에서 받은 파일명/경로로 변경


def load_msft(csv_path=None):
    """우선순위: (1) 주어진 CSV → (2) yfinance 실제 MSFT → (3) SYNTHETIC(코드 확인용, 실제 MSFT 아님)."""
    csv_path = csv_path or MSFT_CSV
    if os.path.exists(csv_path):
        print('[data] CSV:', csv_path)
        return pd.read_csv(csv_path)
    try:
        import yfinance as yf
        d = yf.download('MSFT', start='2015-01-01', end='2024-12-31', auto_adjust=False, progress=False)
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)
        d = d.reset_index()
        if len(d) == 0:
            raise ValueError('empty download')
        print('[data] yfinance(Yahoo Finance) 실제 MSFT 일봉:', d.shape)
        return d
    except Exception as e:
        print('[data] CSV·yfinance 모두 불가(', type(e).__name__, ') → ⚠ SYNTHETIC 데이터: 실제 MSFT가 아님. 코드 동작 확인용')
        rng = np.random.default_rng(0)
        n = 1500
        close = 100 * np.exp(np.cumsum(rng.normal(0.0005, 0.015, n)))
        open_ = close * (1 + rng.normal(0, 0.005, n))
        high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.004, n)))
        low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.004, n)))
        return pd.DataFrame({'Date': pd.bdate_range('2018-01-01', periods=n),
                             'Open': open_, 'High': high, 'Low': low, 'Close': close,
                             'Adj Close': close, 'Volume': rng.integers(10_000_000, 50_000_000, n)})


MSFT_FEATURES = ['Open', 'High', 'Low', 'Close', 'Volume']
MSFT_TARGETS = ['Next_Open', 'Next_Close']     # 출력 2차원 벡터


def make_msft_dataset(df=None, ratios=(0.7, 0.15), inject_nan=False):
    """C01 정답과 동일: 날짜 정렬 → dropna → 다음날 [Open, Close] 라벨 → 시간순 분할(shuffle 금지)."""
    df = load_msft() if df is None else df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    if inject_nan:
        df.iloc[[5, 50, 100], df.columns.get_loc('Close')] = np.nan
    df = df.dropna(subset=MSFT_FEATURES).copy()
    df['Next_Open'] = df['Open'].shift(-1)
    df['Next_Close'] = df['Close'].shift(-1)
    df = df.dropna(subset=MSFT_TARGETS)
    X, y = df.loc[:, MSFT_FEATURES], df.loc[:, MSFT_TARGETS]
    n = len(df)
    i1, i2 = int(n * ratios[0]), int(n * (ratios[0] + ratios[1]))
    return X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:], y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]


def train_msft_quick(epochs=100, loss='mse', verbose=0):
    """C02 정답의 축약판. 다른 문제에서 학습된 모델이 필요할 때 사용."""
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import StandardScaler
    tf.keras.utils.set_random_seed(42)
    X_tr, X_va, X_te, y_tr, y_va, y_te = make_msft_dataset()
    sx, sy = StandardScaler(), StandardScaler()
    Xtr, Xva, Xte = sx.fit_transform(X_tr), sx.transform(X_va), sx.transform(X_te)
    ytr, yva = sy.fit_transform(y_tr), sy.transform(y_va)
    m = Sequential([Input(shape=(Xtr.shape[1],)), Dense(64, activation='relu'),
                    Dense(32, activation='relu'), Dense(2)])
    m.compile(optimizer='adam', loss=loss, metrics=['mae'])
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=32, verbose=verbose,
              callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])
    y_pred = sy.inverse_transform(m.predict(Xte, verbose=0))
    return dict(model=m, history=h, X_test=X_te, y_test=y_te, y_pred=y_pred, scaler_X=sx, scaler_y=sy)


# ---------- Lab2: ETTh1 ----------
def load_etth1():
    path = _csv('ETTh1.csv')
    if os.path.exists(path):
        print('[data] 강의 CSV:', path)
        return pd.read_csv(path)
    url = 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv'
    print('[data] 강의 CSV 없음 → 대체: ETDataset GitHub 원본 ETTh1.csv (행 수가 강의 CSV와 다를 수 있음)')
    return pd.read_csv(url)


# ---------- Lab2: ABB IRB 2400 (datasetIRB2400.csv) ----------
IK_FEATURES = ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
IK_TARGETS = ['q1_out', 'q2_out', 'q3_out', 'q4_out', 'q5_out', 'q6_out']


def load_irb2400():
    """반환: (df, 입력열, 출력열). 강의 CSV 없으면 sklearn linnerud(실제 다중출력 데이터, 20행)로 대체."""
    path = _csv('datasetIRB2400.csv')
    if os.path.exists(path):
        print('[data] 강의 CSV:', path)
        return pd.read_csv(path), IK_FEATURES, IK_TARGETS
    print('[data] 강의 CSV 없음 → 대체: sklearn load_linnerud (입력 3개 → 출력 3개, 20행). IRB2400 아님')
    from sklearn.datasets import load_linnerud
    L = load_linnerud(as_frame=True)
    d = pd.concat([L.data, L.target], axis=1)
    return d, list(L.data.columns), list(L.target.columns)


# ---------- 족보: Fashion-MNIST ----------
def load_fashion(val_size=0.2, seed=42):
    """E01+E02 정답과 동일: 0~1 정규화, (N,28,28,1), train에서만 검증셋 분리(stratify)."""
    from tensorflow.keras.datasets import fashion_mnist
    from sklearn.model_selection import train_test_split
    (x_full, y_full), (x_test, y_test) = fashion_mnist.load_data()
    x_full = x_full.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    x_train, x_val, y_train, y_val = train_test_split(
        x_full, y_full, test_size=val_size, random_state=seed, stratify=y_full)
    return x_train, x_val, x_test, y_train, y_val, y_test


def train_cnn_quick(epochs=3):
    """E05 정답 축약판 (Confusion matrix 등 학습된 CNN이 필요할 때)."""
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
    tf.keras.utils.set_random_seed(42)
    x_tr, x_va, x_te, y_tr, y_va, y_te = load_fashion()
    m = Sequential([Input(shape=(28, 28, 1)),
                    Conv2D(32, (3, 3), activation='relu', padding='same'), MaxPooling2D((2, 2)),
                    Conv2D(64, (3, 3), activation='relu', padding='same'), MaxPooling2D((2, 2)),
                    Flatten(), Dense(128, activation='relu'), Dropout(0.3),
                    Dense(10, activation='softmax')])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=epochs, batch_size=128, verbose=1)
    return m, h, (x_te, y_te)


# ---------- 공통: 학습곡선 (Lab1 p.39, Lab2 p.48) ----------
def plot_history(history, metric=None):
    hist = pd.DataFrame(history.history)
    keys = ['loss'] + ([metric] if metric else [])
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4))
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, keys):
        ax.plot(hist[k], label='Train ' + k)
        if 'val_' + k in hist:
            ax.plot(hist['val_' + k], label='Validation ' + k)
        ax.set_xlabel('epoch'); ax.set_ylabel(k); ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()
    if 'val_loss' in hist:
        print('best epoch (min val_loss):', int(hist['val_loss'].idxmin()) + 1)
    return hist


try:
    import tensorflow as tf
    import keras
    print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
except ImportError:
    print('[info] TensorFlow 없음 (Colab에서는 기본 설치됨)')
print('SETUP 완료. DATA_DIR =', DATA_DIR)

## 문제 목록 (코드북 Part 1 인덱스와 동일 ID)
| ID | 난이도 | 제목 | 근거 |
|---|---|---|---|
| A01 | 하 | 주사위 2개 게임 (randint + if/elif/else) | 족보 직접 등장 (해당 강의자료 미첨부 → 일반 구현) |
| A02 | 중 | 가위바위보 N판 (np.random.choice + 승패 규칙) | 족보 직접 등장 (해당 강의자료 미첨부 → 일반 구현) |
| A03 | 중 | 업다운 숫자 맞히기 (while + break) — 빈칸 채우기 | 족보 직접 등장 (해당 강의자료 미첨부 → 일반 구현) |
| A04 | 상 | 몬테카를로 시뮬레이션: 주사위 합 분포 (벡터화 + np.where + bar) | 강의자료 기반 예상 |
| A05 | 하 | 출력 결과 예측: seed 고정 난수 + boolean indexing | 강의자료 기반 예상 |
| B01 | 하 | loc vs iloc 슬라이싱 출력 예측 | 족보 항목 + 강의자료 근거 |
| B02 | 중 | 결측치 확인·제거 (isna, dropna subset/how, fillna 비교) | 족보 항목 + 강의자료 근거 |
| B03 | 중 | DataFrame 라벨링: 열 이름 지정, 날짜 인덱스 슬라이싱, 라벨 열 생성 | 족보 직접 등장 |
| B04 | 상 | [오류 수정] 결측치 처리 데이터 누수 (Ames Housing) | 강의자료 기반 예상 |
| B05 | 상 | 범주형 OneHotEncoder + 수치형 StandardScaler + concatenate | 강의자료 기반 예상 |
| B06 | 중 | EDA: value_counts, describe, 0값(사실상 결측) 처리, corr heatmap, 클래스별 히스토그램 | 강의자료 기반 예상 |
| C01 | 중 | MSFT 데이터 로드 → 결측치 제거 → 다음날 [Open, Close] 라벨 → 시간순 분할 | 족보 직접 등장 |
| C02 | 상 | 출력 2차원 회귀 MLP (Dense(2) linear, X·y 표준화, inverse_transform) | 족보 직접 등장 |
| C03 | 상 | 사용자 정의 손실함수 (custom MSE · 가중 MSE · Huber) 구현과 적용 | 족보 직접 등장 |
| C04 | 중 | 모델 성능 시각화: 실제값 vs 예측값 (2개 출력 subplots + 산점도 y=x) | 족보 직접 등장 |
| C05 | 중 | 학습곡선 (history.history → DataFrame, loss/val_loss, best epoch) | 족보 항목 + 강의자료 근거 |
| C06 | 상 | [오류 수정] 회귀 코드에 섞인 분류 설정·누수·셔플 오류 | 강의자료 기반 예상 |
| C07 | 상 | 다중 출력 회귀 응용: 역기구학 (pose 6 → joint 6), 출력별 원래 단위 MAE | 강의자료 기반 예상 |
| C08 | 최상 | 시계열 직접 예측: 슬라이딩 윈도우 96→24, Last-Value/Seasonal 기준선, Residual MLP | 강의자료 기반 예상 |
| C09 | 중 | 목표값 로그 변환 (log1p / expm1) — 빈칸 채우기 | 강의자료 기반 예상 |
| D01 | 하 | Pima 이진 분류 MLP + 파라미터 수 계산 | 강의자료 기반 예상 |
| D02 | 중 | sigmoid 이진 분류 → softmax 2클래스 변환 (to_categorical) | 강의자료 기반 예상 |
| D03 | 중 | Iris 다중분류: get_dummies vs LabelEncoder+sparse — 빈칸 채우기 | 강의자료 기반 예상 |
| D04 | 하 | [오류 수정] 라벨 형식과 손실함수 불일치 | 강의자료 기반 예상 |
| D05 | 상 | Wine 분류: stratify 분할 + validation_split + EarlyStopping + ModelCheckpoint + load_model | 강의자료 기반 예상 |
| D06 | 중 | 혼동행렬로 Accuracy·Precision·Recall·F1 직접 계산 + 불균형 데이터 함정 | 강의자료 기반 예상 |
| D07 | 하 | 다중 라벨 분류 (sigmoid + binary_crossentropy + threshold 0.5) — 빈칸 채우기 | 강의자료 기반 예상 |
| E01 | 하 | Fashion-MNIST 불러오기 + 0~1 정규화 + 채널 차원 추가 | 족보 직접 등장 |
| E02 | 중 | 훈련/검증 데이터 분리 (stratify) — test는 손대지 않기 | 족보 직접 등장 |
| E03 | 하 | 훈련 이미지 10개 2×5 그리드 시각화 (클래스 이름 제목) | 족보 직접 등장 |
| E04 | 상 | 훈련 데이터에만 실시간 데이터 증강 2가지 이상 (ImageDataGenerator.flow) | 족보 직접 등장 |
| E05 | 상 | Conv2D 기반 CNN 설계·학습·평가 + 파라미터 수 | 족보 직접 등장 |
| E06 | 상 | Weight decay 적용: (A) kernel_regularizer=l2 (B) AdamW | 족보 직접 등장 |
| E07 | 중 | 과적합 방지: Dropout + EarlyStopping(restore_best_weights) + ModelCheckpoint | 족보 직접 등장 |
| E08 | 중 | Confusion matrix 출력 (ConfusionMatrixDisplay, 정규화, 가장 헷갈린 쌍) | 족보 직접 등장 |
| E09 | 상 | [오류 수정] CNN 파이프라인의 7가지 실수 | 강의자료 기반 예상 |
| E10 | 상 | CNN에 BatchNormalization 추가 (Conv → BN → ReLU) | 강의자료 기반 예상 |
| E11 | 중 | 증강 결과 시각화: 원본 1장 + 증강 9장 (2×5) | 강의자료 기반 예상 |
| E12 | 최상 | [대체 구현] Keras 전처리 레이어로 모델 내부 증강 (RandomFlip/Rotation/Translation/Zoom) | 강의자료 기반 예상 (강의자료 외 대체 구현 포함) |
| F01 | 중 | Optimizer 비교: SGD / Momentum / Nesterov / RMSprop / Adam / AdamW | 강의자료 기반 예상 |
| F02 | 상 | 학습률 스케줄: Warm-up + Cosine decay (LearningRateScheduler) + ReduceLROnPlateau | 강의자료 기반 예상 |
| F03 | 중 | 가중치 초기화 실험: 너무 작은/큰 분산 vs Xavier vs He (NumPy) + Keras initializer | 강의자료 기반 예상 |
| F04 | 하 | 출력층·손실·은닉 활성화 짝 맞추기 — 빈칸 채우기 | 강의자료 기반 예상 |
| F05 | 최상 | NumPy로 GD vs Mini-batch SGD + L2 정규화 직접 구현 (슬라이드 수치 재현) | 강의자료 기반 예상 |
| G01 | 최상 | [통합] Fashion-MNIST CNN 전체 파이프라인 (족보 항목 9개 연결) | 족보 직접 등장 |
| G02 | 최상 | [통합] MSFT 2차원 출력 회귀 (결측치 → 라벨링·슬라이싱 → 사용자 정의 손실 → 시각화) | 족보 직접 등장 |
| G03 | 상 | [통합] NumPy 게임 기록 → Pandas DataFrame 라벨링·슬라이싱·결측치·시각화 | 족보 직접 등장 |
| G04 | 상 | [통합] Pima 이진 분류: 6:2:2 분할 → 표준화 → Dropout·L2 MLP → EarlyStopping → 평가지표·혼동행렬 | 강의자료 기반 예상 |

# A. NumPy 난수 · 조건문 게임
- **A01** [하] 주사위 2개 게임 (randint + if/elif/else)
- **A02** [중] 가위바위보 N판 (np.random.choice + 승패 규칙)
- **A03** [중] 업다운 숫자 맞히기 (while + break) — 빈칸 채우기
- **A04** [상] 몬테카를로 시뮬레이션: 주사위 합 분포 (벡터화 + np.where + bar)
- **A05** [하] 출력 결과 예측: seed 고정 난수 + boolean indexing

## A — 문제

### A01. 주사위 2개 게임 (randint + if/elif/else)
**유형:** NumPy 난수·조건문 / 코드 작성 · **출제 근거:** 족보 직접 등장 (해당 강의자료 미첨부 → 일반 구현) · **난이도:** 하

**문제:** `np.random.seed(42)`로 고정한 뒤 주사위 2개를 10번 굴린다. 매 판마다
- 두 눈이 같으면 `Double!`
- 합이 10 이상이면 `High`
- 그 외는 `Low`

를 출력하고, 마지막에 각 결과의 횟수를 출력하라.

**Search Keywords:** `np.random.seed` · `np.random.randint` · `if elif else` · `for` · `dict count`

**Source:** 족보: "NumPy 난수 생성 및 조건문을 이용한 게임" (NumPy 기초 강의자료는 첨부되지 않음 → 일반 구현)

In [ ]:
# A01 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
np.random.seed(42)
# TODO: 10판 반복, randint로 주사위 2개, 조건 판정, 횟수 집계

### A02. 가위바위보 N판 (np.random.choice + 승패 규칙)
**유형:** NumPy 난수·조건문 / 코드 작성 · **출제 근거:** 족보 직접 등장 (해당 강의자료 미첨부 → 일반 구현) · **난이도:** 중

**문제:** 사용자와 컴퓨터가 모두 `np.random.choice`로 가위·바위·보를 고르는 게임을 20판 진행한다(seed=0).
(1) 판마다 결과(승/무/패) 출력 (2) 승·무·패 횟수와 승률(%) 출력.
※ 사용자 입력을 받는 버전(`input()`)도 주석으로 작성할 것.

**Search Keywords:** `np.random.choice` · `np.random.seed` · `dict` · `if elif` · `input`

**Source:** 족보: "NumPy 난수 생성 및 조건문을 이용한 게임" (일반 구현)

In [ ]:
# A02 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
np.random.seed(0)
options = ['가위', '바위', '보']
beats = {'가위': '보', '바위': '가위', '보': '바위'}   # key가 value를 이김
# TODO

### A03. 업다운 숫자 맞히기 (while + break) — 빈칸 채우기
**유형:** NumPy 난수·조건문 / 빈칸 채우기 · **출제 근거:** 족보 직접 등장 (해당 강의자료 미첨부 → 일반 구현) · **난이도:** 중

**문제:** 1~100 사이 정답을 `np.random.randint`로 만들고, 추측값이 크면 `Down`, 작으면 `Up`, 맞으면 시도 횟수를 출력하고 종료하는 게임의 빈칸을 채워라.
`AUTO_PLAY=True`면 이분탐색으로 자동 추측(시험장에서 input 없이 확인용).

**Search Keywords:** `np.random.randint` · `while True` · `break` · `input` · `int()`

**Source:** 족보: "NumPy 난수 생성 및 조건문을 이용한 게임" (일반 구현)

In [ ]:
# A03 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
np.random.seed(7)
answer = np.random.randint(____, ____)     # 1~100
AUTO_PLAY = True
low, high, tries = 1, 100, 0
while ____:
    guess = (low + high) // 2 if AUTO_PLAY else int(input('1~100: '))
    tries += 1
    if guess ____ answer:
        print(guess, 'Down'); high = guess - 1
    elif guess ____ answer:
        print(guess, 'Up'); low = guess + 1
    else:
        print(f'정답 {answer}! {tries}번 만에 성공')
        ____

### A04. 몬테카를로 시뮬레이션: 주사위 합 분포 (벡터화 + np.where + bar)
**유형:** NumPy 난수·조건문 / 시각화 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** 반복문 없이(벡터화) 주사위 2개를 100,000번 굴려라(seed=1).
(1) 합(2~12)별 경험적 확률을 `np.bincount`로 구하고 이론 확률(예: 합 7 → 6/36)과 표로 비교
(2) `np.where`로 합 ≥ 10이면 "WIN", 아니면 "LOSE" 라벨을 만들고 WIN 비율 출력 (이론값 6/36)
(3) 경험적 확률 막대그래프 + 이론값 점 그래프.

**Search Keywords:** `np.random.randint size=(n,2)` · `sum(axis=1)` · `np.bincount` · `np.where` · `plt.bar` · `pd.DataFrame`

**Source:** 족보 게임 항목의 응용(벡터화). 강의자료 근거 없음 → 일반 구현

In [ ]:
# A04 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(1)
N = 100_000
# TODO: rolls (N,2) → sums → bincount → 비교표 → np.where 라벨 → 그래프

### A05. 출력 결과 예측: seed 고정 난수 + boolean indexing
**유형:** NumPy 난수·조건문 / 출력 예측 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 하

**문제:** 다음 코드의 출력 5줄을 예측하라 (실행 전 손으로 적고 확인).

**Search Keywords:** `np.random.seed` · `boolean indexing` · `np.where` · `argmax`

**Source:** 족보 게임 항목 대비 기초 문법 확인 (일반 구현)

In [ ]:
# A05 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
np.random.seed(0)
a = np.random.randint(1, 7, size=6)
print(a)
print(a[a >= 4])
print(np.where(a % 2 == 0, 'even', 'odd'))
print(a.sum(), a.max(), a.argmax())
print((a == 6).sum(), np.unique(a))

## A — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### A01 정답 — 주사위 2개 게임 (randint + if/elif/else)
**Expected Output:** 
```text
 1판: 4, 5 (합  9) → Low
 2판: 3, 5 (합  8) → Low
 3판: 5, 2 (합  7) → Low
 4판: 3, 3 (합  6) → Double!
 5판: 3, 5 (합  8) → Low
 6판: 4, 3 (합  7) → Low
 7판: 6, 5 (합 11) → High
 8판: 2, 4 (합  6) → Low
 9판: 6, 6 (합 12) → Double!
10판: 2, 4 (합  6) → Low
{'Double!': 2, 'High': 1, 'Low': 7}
```

**Modification Point:** 주사위 범위 `randint(1, 7)`의 7(상한 미포함), 판 수 `range(1, 11)`, 판정 조건 순서(더블을 먼저 검사해야 더블+High 중복 방지).

In [ ]:
# ===== A01 정답 =====
import numpy as np
np.random.seed(42)                         # seed 고정 → 매번 같은 결과
counts = {'Double!': 0, 'High': 0, 'Low': 0}
for game in range(1, 11):
    d1, d2 = np.random.randint(1, 7, size=2)   # 1~6 (high는 미포함 → 7)
    if d1 == d2:
        result = 'Double!'
    elif d1 + d2 >= 10:
        result = 'High'
    else:
        result = 'Low'
    counts[result] += 1
    print(f'{game:2d}판: {d1}, {d2} (합 {d1 + d2:2d}) → {result}')
print(counts)

### A02 정답 — 가위바위보 N판 (np.random.choice + 승패 규칙)
**Expected Output:** 
```text
 1판 user=가위 com=바위 → 패
 2판 user=가위 com=바위 → 패
 3판 user=바위 com=보 → 패
 4판 user=가위 com=보 → 승
 5판 user=가위 com=가위 → 무
 6판 user=가위 com=보 → 승
 7판 user=바위 com=보 → 패
 8판 user=보 com=가위 → 패
 9판 user=바위 com=바위 → 무
10판 user=바위 com=바위 → 무
11판 user=가위 com=바위 → 패
12판 user=가위 com=가위 → 무
13판 user=바위 com=보 → 패
14판 user=가위 com=보 → 승
15판 user=가위 com=바위 → 패
16판 user=바위 com=보 → 패
17판 user=가위 com=바위 → 패
18판 user=바위 com=바위 → 무
19판 user=가위 com=보 → 승
20판 user=가위 com=보 → 승
{'승': 5, '무': 5, '패': 10} 승률 25.0%
```

**Modification Point:** 판 수 `N`, 규칙 딕셔너리 `beats`, 승률 분모(무승부 제외 시 `record["승"]+record["패"]`).

In [ ]:
# ===== A02 정답 =====
import numpy as np
np.random.seed(0)
options = ['가위', '바위', '보']
beats = {'가위': '보', '바위': '가위', '보': '바위'}   # key가 value를 이김
record = {'승': 0, '무': 0, '패': 0}
N = 20
for i in range(1, N + 1):
    user = np.random.choice(options)
    # user = input('가위/바위/보 입력: ').strip()   # ← 직접 입력 버전
    com = np.random.choice(options)
    if user == com:
        res = '무'
    elif beats[user] == com:
        res = '승'
    else:
        res = '패'
    record[res] += 1
    print(f'{i:2d}판 user={user} com={com} → {res}')
print(record, f"승률 {record['승'] / N * 100:.1f}%")

### A03 정답 — 업다운 숫자 맞히기 (while + break) — 빈칸 채우기
**Expected Output:** 
```text
50 Down
25 Up
37 Up
43 Up
46 Up
정답 48! 6번 만에 성공
```

**Modification Point:** 범위(`randint(1, 101)`), 최대 시도 제한이 있으면 `while tries < MAX:` + `else:` 실패 메시지.

In [ ]:
# ===== A03 정답 =====
import numpy as np
np.random.seed(7)
answer = np.random.randint(1, 101)         # 1~100 (101 미포함)
AUTO_PLAY = True                           # False → input()으로 직접 플레이
low, high, tries = 1, 100, 0
while True:
    guess = (low + high) // 2 if AUTO_PLAY else int(input('1~100: '))
    tries += 1
    if guess > answer:
        print(guess, 'Down'); high = guess - 1
    elif guess < answer:
        print(guess, 'Up'); low = guess + 1
    else:
        print(f'정답 {answer}! {tries}번 만에 성공')
        break

### A04 정답 — 몬테카를로 시뮬레이션: 주사위 합 분포 (벡터화 + np.where + bar)
**Expected Output:** 
```text
 sum  empirical  theory
   2     0.0278  0.0278
   3     0.0547  0.0556
   4     0.0852  0.0833
   5     0.1108  0.1111
   6     0.1383  0.1389
   7     0.1651  0.1667
   8     0.1394  0.1389
   9     0.1120  0.1111
  10     0.0833  0.0833
  11     0.0552  0.0556
  12     0.0282  0.0278
WIN 비율: 0.1667 | 이론: 0.1667
```

**Modification Point:** `size=(N, k)`로 주사위 개수 변경 시 `minlength`와 이론값 계산 변경. 조건 라벨은 `np.where(조건, A, B)` 또는 다중 조건은 `np.select`.

In [ ]:
# ===== A04 정답 =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(1)
N = 100_000
rolls = np.random.randint(1, 7, size=(N, 2))     # (N, 2)
sums = rolls.sum(axis=1)                          # 행 방향 합
emp = np.bincount(sums, minlength=13)[2:] / N     # index 2~12
theory = np.array([min(s - 1, 13 - s) for s in range(2, 13)]) / 36
table = pd.DataFrame({'sum': range(2, 13), 'empirical': emp.round(4), 'theory': theory.round(4)})
print(table.to_string(index=False))
labels = np.where(sums >= 10, 'WIN', 'LOSE')
print('WIN 비율:', (labels == 'WIN').mean().round(4), '| 이론:', round(6 / 36, 4))
plt.bar(table['sum'], table['empirical'], label='empirical')
plt.plot(table['sum'], table['theory'], 'ro', label='theory')
plt.xlabel('sum of two dice'); plt.ylabel('probability'); plt.legend(); plt.show()

### A05 정답 — 출력 결과 예측: seed 고정 난수 + boolean indexing
**Expected Output:** 
```text
[5 6 1 4 4 4]
[5 6 4 4 4]
['odd' 'even' 'odd' 'even' 'even' 'even']
24 6 1
1 [1 4 5 6]
```

**Modification Point:** `np.random.default_rng(0).integers(1, 7, 6)`은 다른 난수 생성기 → 같은 seed라도 값이 다름.

In [ ]:
# ===== A05 정답 =====
import numpy as np
np.random.seed(0)
a = np.random.randint(1, 7, size=6)
print(a)                                  # 같은 seed → 같은 배열
print(a[a >= 4])                          # 조건 True인 원소만
print(np.where(a % 2 == 0, 'even', 'odd'))
print(a.sum(), a.max(), a.argmax())       # argmax: 최댓값의 첫 index
print((a == 6).sum(), np.unique(a))       # True 개수, 정렬된 고유값

# B. Pandas 전처리 · 라벨링 · 슬라이싱 · 결측치
- **B01** [하] loc vs iloc 슬라이싱 출력 예측
- **B02** [중] 결측치 확인·제거 (isna, dropna subset/how, fillna 비교)
- **B03** [중] DataFrame 라벨링: 열 이름 지정, 날짜 인덱스 슬라이싱, 라벨 열 생성
- **B04** [상] [오류 수정] 결측치 처리 데이터 누수 (Ames Housing)
- **B05** [상] 범주형 OneHotEncoder + 수치형 StandardScaler + concatenate
- **B06** [중] EDA: value_counts, describe, 0값(사실상 결측) 처리, corr heatmap, 클래스별 히스토그램

## B — 문제

### B01. loc vs iloc 슬라이싱 출력 예측
**유형:** Pandas 슬라이싱 / 출력 예측 · **출제 근거:** 족보 항목 + 강의자료 근거 · **난이도:** 하

**문제:** 다음 DataFrame에서 각 print 결과를 예측하라. 특히 `loc`(라벨, **끝 포함**)와 `iloc`(위치, **끝 미포함**)의 차이를 확인하라.

**Search Keywords:** `DataFrame.loc` · `DataFrame.iloc` · `boolean indexing` · `df.iloc[:, 0:8]`

**Source:** Lecture6_Lab1 p.13 (iloc), 족보 "Pandas DataFrame 라벨링 및 슬라이싱"

In [ ]:
# B01 연습 셀 — 빈칸(____)/TODO를 채워 실행
import pandas as pd
df = pd.DataFrame({'Open': [10, 11, 12, 13, 14], 'Close': [11, 12, 11, 14, 15],
                   'Volume': [100, 200, 150, 300, 250]},
                  index=['d1', 'd2', 'd3', 'd4', 'd5'])
print(df.loc['d2':'d4', 'Open':'Close'])
print(df.iloc[1:4, 0:2])
print(df.iloc[:, -1].tolist())
print(df.loc[df['Close'] > df['Open'], 'Volume'].sum())
print(df.iloc[[0, -1]])

### B02. 결측치 확인·제거 (isna, dropna subset/how, fillna 비교)
**유형:** 데이터 전처리 / 코드 작성 · **출제 근거:** 족보 항목 + 강의자료 근거 · **난이도:** 중

**문제:** 아래 DataFrame에 대해
(1) 열별 결측치 개수를 내림차순 출력 (2) 결측치가 하나라도 있는 행 제거 후 shape
(3) `Close` 열이 결측인 행만 제거 (4) 모든 값이 결측인 행만 제거 (5) `Volume` 결측은 0, `Close` 결측은 중앙값으로 채우기
(6) 제거 후 인덱스를 0부터 다시 매기기.

**Search Keywords:** `isna().sum()` · `sort_values` · `dropna()` · `dropna(subset=)` · `dropna(how='all')` · `fillna` · `reset_index(drop=True)`

**Source:** Lecture11_Lab2 p.7 (isna().sum().sort_values), p.9 (fillna), 족보 "결측치 제거"

In [ ]:
# B02 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np, pandas as pd
df = pd.DataFrame({'Open':   [10, 11, np.nan, 13, np.nan, 15],
                   'Close':  [11, np.nan, 12, 14, np.nan, 16],
                   'Volume': [100, 200, 150, np.nan, np.nan, 300]})
# TODO (1)~(6)

### B03. DataFrame 라벨링: 열 이름 지정, 날짜 인덱스 슬라이싱, 라벨 열 생성
**유형:** Pandas 라벨링·슬라이싱 / 코드 작성 · **출제 근거:** 족보 직접 등장 · **난이도:** 중

**문제:** 헤더 없는 주가 데이터가 주어졌다.
(1) 열 이름을 `Date, Open, High, Low, Close, Volume`으로 지정 (2) `Date`를 datetime으로 바꿔 인덱스로 설정
(3) 2024-01-03 ~ 2024-01-05 구간의 `Open, Close`만 슬라이싱 (4) 다음날 종가가 오늘보다 높으면 1, 아니면 0인 `Up` 라벨 열 생성(마지막 행 처리 주의)
(5) `Close`를 3구간(`low/mid/high`)으로 나눈 `Level` 라벨 생성 (6) `Close`→`Price` 열 이름 변경.

**Search Keywords:** `df.columns =` · `header=None` · `pd.to_datetime` · `set_index` · `loc['2024-01-03':'2024-01-05']` · `shift(-1)` · `astype(int)` · `pd.cut` · `rename(columns=)`

**Source:** 족보 "Pandas DataFrame 라벨링 및 슬라이싱", Lecture6_Lab1 p.32~33 (header=None, iloc)

In [ ]:
# B03 연습 셀 — 빈칸(____)/TODO를 채워 실행
import io, pandas as pd
raw = """2024-01-02,370.0,376.0,366.5,370.9,25000000
2024-01-03,369.0,373.3,368.5,370.6,23000000
2024-01-04,370.7,373.1,367.2,367.9,20900000
2024-01-05,368.9,372.1,366.5,367.8,20900000
2024-01-08,369.3,375.2,369.0,374.7,23100000"""
df = pd.read_csv(io.StringIO(raw), header=None)     # 헤더 없는 CSV (Lab1 wine.csv와 같은 상황)
# TODO (1)~(6)

### B04. [오류 수정] 결측치 처리 데이터 누수 (Ames Housing)
**유형:** 데이터 전처리 / 오류 수정 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** 아래 코드는 실행은 되지만 **잘못된 실습 관행 3가지**가 있다. 찾아서 Lab2 방식으로 수정하라.
힌트: 입력에 넣으면 안 되는 열, 분할 전 통계 계산, 숫자로 저장된 범주형 변수.

**Search Keywords:** `drop(columns=)` · `train_test_split` · `median()` · `mode()[0]` · `fillna` · `astype(str)` · `data leakage`

**Source:** Lecture11_Lab2 p.6 (Id 제외), p.8 (astype(str), 60:20:20 split), p.9 (train median/mode)

In [ ]:
# B04 연습 셀 — 빈칸(____)/TODO를 채워 실행
# ❌ 버그 있는 코드 — 실행 전 0번 SETUP 셀 실행
from sklearn.model_selection import train_test_split
df = load_ames()
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())   # 버그?
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
X = df.drop(columns=['SalePrice'])                                         # 버그?
y = df['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.dtypes['MSSubClass'])                                        # 버그?

### B05. 범주형 OneHotEncoder + 수치형 StandardScaler + concatenate
**유형:** 데이터 전처리 / 코드 작성 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** B04로 결측치를 처리한 Ames 데이터에 대해
(1) object 열은 `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`로 인코딩 (train만 `fit_transform`)
(2) 수치 열은 `StandardScaler` (train만 `fit_transform`, val/test는 `transform`)
(3) 두 결과를 `np.concatenate(axis=1)`로 합쳐 신경망 입력을 만들고 shape 출력
(4) 목표값 `SalePrice`도 train 기준으로 표준화.

**Search Keywords:** `OneHotEncoder` · `handle_unknown='ignore'` · `sparse_output=False` · `select_dtypes(include='object')` · `StandardScaler` · `fit_transform` · `transform` · `np.concatenate` · `reshape(-1, 1)`

**Source:** Lecture11_Lab2 p.10 (OneHotEncoder), p.11 (StandardScaler, concatenate), Lecture9 p.20 (fit_transform vs transform, target scaling)

In [ ]:
# B05 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 0번 SETUP 실행 후. prepare_ames()를 쓰지 말고 직접 작성해 볼 것
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# TODO

### B06. EDA: value_counts, describe, 0값(사실상 결측) 처리, corr heatmap, 클래스별 히스토그램
**유형:** 데이터 전처리 / 시각화 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** Pima 데이터에 대해
(1) 라벨 `diabetes` 비율(%) (2) `describe()` (3) `plasma, pressure, thickness, insulin, bmi` 열의 0 개수 → 0을 NaN으로 바꾼 뒤 결측 개수 출력
(4) 상관행렬 heatmap (annot) 및 `diabetes`와 상관이 가장 큰 특성 2개 (5) `plasma`의 클래스별 히스토그램 (`density=True`).

**Search Keywords:** `value_counts(normalize=True)` · `describe` · `replace(0, np.nan)` · `df.corr()` · `sns.heatmap` · `plt.hist` · `histtype='step'` · `density=True`

**Source:** Lecture6_Lab1 p.6 (value_counts), p.7 (describe, 0=결측 가능), p.8~10 (corr, heatmap), p.11 (hist)

In [ ]:
# B06 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 0번 SETUP 실행 후
mount_drive()
df = load_pima()
# TODO (1)~(5)

## B — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### B01 정답 — loc vs iloc 슬라이싱 출력 예측
**Expected Output:** 
```text
    Open  Close
d2    11     12
d3    12     11
d4    13     14
    Open  Close
d2    11     12
d3    12     11
d4    13     14
[100, 200, 150, 300, 250]
850
    Open  Close  Volume
d1    10     11     100
d5    14     15     250
```

**Modification Point:** 열 개수가 다른 데이터면 `iloc[:, 0:8]`의 8을 `iloc[:, :-1]`(마지막 열 제외)로 일반화.

In [ ]:
# ===== B01 정답 =====
import pandas as pd
df = pd.DataFrame({'Open': [10, 11, 12, 13, 14], 'Close': [11, 12, 11, 14, 15],
                   'Volume': [100, 200, 150, 300, 250]},
                  index=['d1', 'd2', 'd3', 'd4', 'd5'])
print(df.loc['d2':'d4', 'Open':'Close'])    # loc: 'd4', 'Close' 포함 → 3행 2열
print(df.iloc[1:4, 0:2])                    # iloc: 4, 2 미포함 → 같은 3행 2열
print(df.iloc[:, -1].tolist())              # 마지막 열 = Volume
print(df.loc[df['Close'] > df['Open'], 'Volume'].sum())   # 조건 행의 Volume 합
print(df.iloc[[0, -1]])                     # 첫 행 + 마지막 행
# 강의 Lab1 p.13: X = df.iloc[:, 0:8] (0~7열), y = df.iloc[:, 8] (8열)

### B02 정답 — 결측치 확인·제거 (isna, dropna subset/how, fillna 비교)
**Expected Output:** 
```text
Open      2
Close     2
Volume    2
dtype: int64
전체 결측 개수: 6
(2) (2, 3)
(3) (4, 3)
(4) (5, 3)
(5)
    Open  Close  Volume
0  10.0   11.0   100.0
1  11.0   13.0   200.0
2   NaN   12.0   150.0
3  13.0   14.0     0.0
4   NaN   13.0     0.0
5  15.0   16.0   300.0
(6)
    Open  Close  Volume
0  10.0   11.0   100.0
1  15.0   16.0   300.0
```

**Modification Point:** 학습/테스트로 나눈 뒤 채울 때는 **train 통계(중앙값·최빈값)**로 val/test도 채움 → B04 참고. 시계열은 `ffill()`도 가능.

In [ ]:
# ===== B02 정답 =====
import numpy as np, pandas as pd
df = pd.DataFrame({'Open':   [10, 11, np.nan, 13, np.nan, 15],
                   'Close':  [11, np.nan, 12, 14, np.nan, 16],
                   'Volume': [100, 200, 150, np.nan, np.nan, 300]})
print(df.isna().sum().sort_values(ascending=False))          # (1)
print('전체 결측 개수:', df.isna().sum().sum())
d2 = df.dropna();                       print('(2)', d2.shape)
d3 = df.dropna(subset=['Close']);       print('(3)', d3.shape)
d4 = df.dropna(how='all');              print('(4)', d4.shape)
d5 = df.copy()
d5['Volume'] = d5['Volume'].fillna(0)
d5['Close'] = d5['Close'].fillna(d5['Close'].median())
print('(5)\n', d5)
d6 = df.dropna().reset_index(drop=True); print('(6)\n', d6)

### B03 정답 — DataFrame 라벨링: 열 이름 지정, 날짜 인덱스 슬라이싱, 라벨 열 생성
**Expected Output:** 
```text
             Open  Close
Date                    
2024-01-03  369.0  370.6
2024-01-04  370.7  367.9
2024-01-05  368.9  367.8
             Open   High    Low  Price    Volume  Up Level
Date                                                      
2024-01-02  370.0  376.0  366.5  370.9  25000000   0  high
2024-01-03  369.0  373.3  368.5  370.6  23000000   0  high
2024-01-04  370.7  373.1  367.2  367.9  20900000   0   low
2024-01-05  368.9  372.1  366.5  367.8  20900000   1   low
```

**Modification Point:** 라벨 기준이 "수익률 1% 이상"이면 `df["Close"].pct_change().shift(-1) > 0.01`. 행 라벨(index) 지정은 `df.index = [...]` 또는 `set_index`.

In [ ]:
# ===== B03 정답 =====
import io, pandas as pd
raw = """2024-01-02,370.0,376.0,366.5,370.9,25000000
2024-01-03,369.0,373.3,368.5,370.6,23000000
2024-01-04,370.7,373.1,367.2,367.9,20900000
2024-01-05,368.9,372.1,366.5,367.8,20900000
2024-01-08,369.3,375.2,369.0,374.7,23100000"""
df = pd.read_csv(io.StringIO(raw), header=None)
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']            # (1) 열 라벨
df['Date'] = pd.to_datetime(df['Date'])                                     # (2)
df = df.set_index('Date')
print(df.loc['2024-01-03':'2024-01-05', ['Open', 'Close']])                 # (3) 날짜 라벨 슬라이싱(끝 포함)
next_close = df['Close'].shift(-1)                                          # (4) 다음날 종가
df['Up'] = (next_close > df['Close']).astype(int)
df = df[next_close.notna()].copy()                                          #     다음날이 없는 마지막 행 제거
df['Level'] = pd.cut(df['Close'], bins=3, labels=['low', 'mid', 'high'])    # (5)
df = df.rename(columns={'Close': 'Price'})                                  # (6)
print(df)

### B04 정답 — [오류 수정] 결측치 처리 데이터 누수 (Ames Housing)
**Expected Output:** train (876, 79) / val (292, 79) / test (292, 79), 결측 0 (강의 CSV 기준. 대체 데이터는 열 수가 다를 수 있음)

**Modification Point:** 결측 열 목록(`ABSENCE_CAT_COLS`, `ABSENCE_NUM_COLS`)은 0번 SETUP에 정의. 다른 데이터셋이면 `isna().sum()`으로 목록을 다시 만든다.

In [ ]:
# ===== B04 정답 =====
from sklearn.model_selection import train_test_split
mount_drive()
df = load_ames()
X = df.drop(columns=['Id', 'SalePrice']).copy()          # 수정1: Id는 가격과 무관 → 제외
y = df['SalePrice'].copy()
X['MSSubClass'] = X['MSSubClass'].astype(str)            # 수정3: 숫자형 범주 코드 → 문자열
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42)  # 60:20:20
X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()
for data in [X_train, X_val, X_test]:                    # '없음'을 뜻하는 결측
    data[ABSENCE_CAT_COLS] = data[ABSENCE_CAT_COLS].fillna('None')
    data[ABSENCE_NUM_COLS] = data[ABSENCE_NUM_COLS].fillna(0)
lot_frontage_median = X_train['LotFrontage'].median()    # 수정2: 분할 후 train 통계만 사용
electrical_mode = X_train['Electrical'].mode()[0]
for data in [X_train, X_val, X_test]:
    data['LotFrontage'] = data['LotFrontage'].fillna(lot_frontage_median)
    data['Electrical'] = data['Electrical'].fillna(electrical_mode)
for name, data in [('train', X_train), ('val', X_val), ('test', X_test)]:
    print(name, data.shape, 'missing:', int(data.isna().sum().sum()))

### B05 정답 — 범주형 OneHotEncoder + 수치형 StandardScaler + concatenate
**Expected Output:** 강의 CSV 기준: cat (876, 275), input (876, 310)  ← Lab2 p.10, p.14의 "310 → Dense(64)"와 일치. y_train_scaled mean≈0, std≈1

**Modification Point:** MinMax로 바꾸려면 `MinMaxScaler()`. sklearn<1.2이면 `sparse_output` 대신 `sparse=False`. 1차원 y는 `reshape(-1, 1)` 후 스케일링.

In [ ]:
# ===== B05 정답 =====
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
mount_drive()
df = load_ames()
# --- B04와 동일한 결측치 처리 ---
X = df.drop(columns=['Id', 'SalePrice']).copy(); y = df['SalePrice'].astype(float).copy()
X['MSSubClass'] = X['MSSubClass'].astype(str)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42)
X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()
for data in [X_train, X_val, X_test]:
    data[ABSENCE_CAT_COLS] = data[ABSENCE_CAT_COLS].fillna('None')
    data[ABSENCE_NUM_COLS] = data[ABSENCE_NUM_COLS].fillna(0)
med, mode = X_train['LotFrontage'].median(), X_train['Electrical'].mode()[0]
for data in [X_train, X_val, X_test]:
    data['LotFrontage'] = data['LotFrontage'].fillna(med); data['Electrical'] = data['Electrical'].fillna(mode)
# --- (1) One-Hot ---
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
for data in [X_train, X_val, X_test]:
    data[cat_cols] = data[cat_cols].astype(str)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[cat_cols])        # train: 범주 학습
X_val_cat = encoder.transform(X_val[cat_cols])                # val/test: 같은 매핑
X_test_cat = encoder.transform(X_test[cat_cols])
# --- (2) Standardization ---
num_cols = X_train.select_dtypes(include='number').columns
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_val_num = scaler.transform(X_val[num_cols])
X_test_num = scaler.transform(X_test[num_cols])
# --- (3) concatenate ---
X_train_input = np.concatenate([X_train_num, X_train_cat], axis=1)
X_val_input = np.concatenate([X_val_num, X_val_cat], axis=1)
X_test_input = np.concatenate([X_test_num, X_test_cat], axis=1)
print('cat:', X_train_cat.shape, 'num:', X_train_num.shape, 'input:', X_train_input.shape, X_val_input.shape, X_test_input.shape)
# --- (4) target scaling (2D로 reshape 필수) ---
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1))
print('y_train_scaled mean/std:', y_train_scaled.mean().round(3), y_train_scaled.std().round(3))
# 예측 복원: y_pred = y_scaler.inverse_transform(model.predict(X_test_input))

### B06 정답 — EDA: value_counts, describe, 0값(사실상 결측) 처리, corr heatmap, 클래스별 히스토그램
**Expected Output:** (1) 0: 65.1, 1: 34.9  (4) plasma ≈ 0.47, bmi ≈ 0.29 (Lab1 p.10)  (3) insulin, thickness에 0이 매우 많음

**Modification Point:** 다른 열 히스토그램은 `negative["bmi"]`로 열 이름만 변경. 클래스 크기가 다르면 `density=True` (Lab1 p.11).

In [ ]:
# ===== B06 정답 =====
import seaborn as sns
mount_drive()
df = load_pima()
print((df['diabetes'].value_counts(normalize=True) * 100).round(1))       # (1) 65.1 / 34.9
print(df.describe().round(2))                                              # (2)
zero_cols = ['plasma', 'pressure', 'thickness', 'insulin', 'bmi']
print('0 개수:\n', (df[zero_cols] == 0).sum())                            # (3)
df_nan = df.copy()
df_nan[zero_cols] = df_nan[zero_cols].replace(0, np.nan)
print('NaN 개수:\n', df_nan.isna().sum())
corr = df.corr()                                                           # (4)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, linewidths=0.5, vmin=-1, vmax=1, cmap='coolwarm', annot=True, fmt='.2f')
plt.show()
print(corr['diabetes'].drop('diabetes').sort_values(ascending=False).head(2))   # plasma 0.47, bmi 0.29
negative, positive = df[df['diabetes'] == 0], df[df['diabetes'] == 1]          # (5)
plt.hist([negative['plasma'], positive['plasma']], bins=30, density=True,
         histtype='step', linewidth=2, label=['negative', 'positive'])
plt.xlabel('plasma'); plt.legend(); plt.show()

# C. 회귀 (MSFT · 2차원 출력 · 사용자 정의 손실 · Lab2)
- **C01** [중] MSFT 데이터 로드 → 결측치 제거 → 다음날 [Open, Close] 라벨 → 시간순 분할
- **C02** [상] 출력 2차원 회귀 MLP (Dense(2) linear, X·y 표준화, inverse_transform)
- **C03** [상] 사용자 정의 손실함수 (custom MSE · 가중 MSE · Huber) 구현과 적용
- **C04** [중] 모델 성능 시각화: 실제값 vs 예측값 (2개 출력 subplots + 산점도 y=x)
- **C05** [중] 학습곡선 (history.history → DataFrame, loss/val_loss, best epoch)
- **C06** [상] [오류 수정] 회귀 코드에 섞인 분류 설정·누수·셔플 오류
- **C07** [상] 다중 출력 회귀 응용: 역기구학 (pose 6 → joint 6), 출력별 원래 단위 MAE
- **C08** [최상] 시계열 직접 예측: 슬라이딩 윈도우 96→24, Last-Value/Seasonal 기준선, Residual MLP
- **C09** [중] 목표값 로그 변환 (log1p / expm1) — 빈칸 채우기

## C — 문제

### C01. MSFT 데이터 로드 → 결측치 제거 → 다음날 [Open, Close] 라벨 → 시간순 분할
**유형:** 회귀 데이터 전처리 / 코드 작성 · **출제 근거:** 족보 직접 등장 · **난이도:** 중

**문제:** MSFT 일봉 CSV(Date, Open, High, Low, Close, Adj Close, Volume)를 불러와
(1) `Date`를 datetime으로 바꿔 정렬·인덱스 설정 (2) 결측치 개수 확인 후 제거(연습용으로 결측 3개 삽입)
(3) 입력 X = 오늘의 `Open, High, Low, Close, Volume`, 출력 y = **다음날** `Open, Close` (2차원)
(4) 시간 순서를 유지하여 train 70% / val 15% / test 15%로 분할하고 각 shape와 기간 출력.

**Search Keywords:** `pd.read_csv` · `pd.to_datetime` · `sort_values` · `set_index` · `isna().sum()` · `dropna` · `shift(-1)` · `df.loc[:, cols]` · `iloc` · `shuffle=False`

**Source:** 족보 "MSFT 등 별도 데이터셋을 이용한 회귀", "결측치 제거", "출력이 2차원 벡터인 회귀"; Lecture10 p.16 (시가·종가·저가·고가·거래량 → 다음날 시가 예시)

In [ ]:
# C01 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 0번 SETUP 실행 후
mount_drive()
df = load_msft()          # 시험: pd.read_csv('주어진 경로')
# TODO

### C02. 출력 2차원 회귀 MLP (Dense(2) linear, X·y 표준화, inverse_transform)
**유형:** 회귀 / 모델 구현 · **출제 근거:** 족보 직접 등장 · **난이도:** 상

**문제:** C01 데이터로 다음날 `[Open, Close]`를 동시에 예측하는 MLP를 구현하라.
(1) X, y 모두 train 기준 `StandardScaler` (2) `Input(5) → Dense(64, relu) → Dense(32, relu) → Dense(2)` (출력 활성화 없음)
(3) `loss='mse', metrics=['mae']`, EarlyStopping(val_loss, patience=10, restore_best_weights=True)
(4) test 예측을 원래 달러 단위로 복원하여 출력별 MAE 출력.

**Search Keywords:** `Dense(2)` · `activation='linear'` · `StandardScaler` · `fit_transform` · `inverse_transform` · `loss='mse'` · `metrics=['mae']` · `EarlyStopping` · `restore_best_weights` · `validation_data` · `mean_absolute_error`

**Source:** 족보 "출력이 2차원 벡터인 회귀"; Lecture9 p.13 (회귀 출력 Dense linear), p.20 (target scaling, inverse_transform); Lecture11_Lab2 p.14, p.47~49 (다중 출력 MLP)

In [ ]:
# C02 연습 셀 — 빈칸(____)/TODO를 채워 실행
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
# TODO

### C03. 사용자 정의 손실함수 (custom MSE · 가중 MSE · Huber) 구현과 적용
**유형:** 회귀 / 사용자 정의 손실 · **출제 근거:** 족보 직접 등장 · **난이도:** 상

**문제:** (1) `custom_mse(y_true, y_pred)`를 TensorFlow 연산으로 직접 구현하고 Keras 내장 MSE와 값이 같은지 확인
(2) 2차원 출력 중 `Next_Close` 오차에 2배 가중치를 주는 `weighted_mse` 구현 (함수 팩토리 형태)
(3) δ=1의 Huber 손실 직접 구현 (4) C02 모델을 (2)의 손실로 compile·학습하고 MAE 비교
(5) 사용자 정의 손실 모델 저장/불러오기 방법 작성.

**Search Keywords:** `custom loss function` · `def loss(y_true, y_pred)` · `tf.reduce_mean` · `tf.square` · `tf.cast` · `tf.where` · `axis=-1` · `model.compile(loss=함수)` · `custom_objects` · `compile=False`

**Source:** 족보 "사용자 정의 손실함수"; Lecture4 p.13·15 (손실 정의 수식), Lecture5 p.15 (ℒ(y, ŷ)), Lecture6_Lab1 p.35 (load_model)

In [ ]:
# C03 연습 셀 — 빈칸(____)/TODO를 채워 실행
import tensorflow as tf
def custom_mse(y_true, y_pred):
    ____
def make_weighted_mse(weights):
    ____

### C04. 모델 성능 시각화: 실제값 vs 예측값 (2개 출력 subplots + 산점도 y=x)
**유형:** 시각화 / 회귀 평가 · **출제 근거:** 족보 직접 등장 · **난이도:** 중

**문제:** 학습된 2출력 회귀 모델로
(1) test 구간에서 `Next_Open`, `Next_Close`의 실제값/예측값 선 그래프를 위아래 2개 subplot으로 그림
(2) 출력별 실제값-예측값 산점도와 y=x 기준선 (3) 잔차(실제-예측) 히스토그램.

**Search Keywords:** `plt.subplots(2, 1)` · `sharex=True` · `ax.plot` · `ax.scatter` · `y=x line` · `plt.hist` · `tight_layout` · `legend`

**Source:** 족보 "모델 성능 시각화"; Lecture4 p.20 (scatter + plot 예측), Lecture6_Lab1 p.39

In [ ]:
# C04 연습 셀 — 빈칸(____)/TODO를 채워 실행
# C02를 실행했으면 reg_model 재사용, 아니면 train_msft_quick()
res = train_msft_quick()
# TODO

### C05. 학습곡선 (history.history → DataFrame, loss/val_loss, best epoch)
**유형:** 시각화 / 학습 과정 · **출제 근거:** 족보 항목 + 강의자료 근거 · **난이도:** 중

**문제:** `model.fit`이 반환한 `history`로
(1) `pd.DataFrame(history.history)` 출력 (2) train/val loss, train/val mae 곡선 (3) val_loss 최소 epoch 출력
(4) 그래프를 보고 과적합 시작 지점을 판단하는 기준을 주석으로 작성.

**Search Keywords:** `history.history` · `pd.DataFrame(history.history)` · `hist_df['val_loss']` · `idxmin` · `plt.plot` · `np.arange(len())` · `plt.ylim`

**Source:** Lecture6_Lab1 p.37~40 (hist_df, 학습곡선), Lecture10 p.3 (learning curves), Lecture11_Lab2 p.48

In [ ]:
# C05 연습 셀 — 빈칸(____)/TODO를 채워 실행
res = train_msft_quick()
history = res['history']
# TODO

### C06. [오류 수정] 회귀 코드에 섞인 분류 설정·누수·셔플 오류
**유형:** 회귀 / 오류 수정 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** 다음 MSFT 2출력 회귀 코드에서 **오류 5가지**를 찾아 수정하라.

**Search Keywords:** `Dense(2)` · `linear` · `loss='mse'` · `metrics=['mae']` · `shuffle=False` · `fit_transform on train` · `inverse_transform`

**Source:** Lecture4 p.3~4 (회귀 ≠ 분류), Lecture9 p.13 (출력층 선택), p.20 (누수 방지), 족보 회귀 항목

In [ ]:
# C06 연습 셀 — 빈칸(____)/TODO를 채워 실행
# ❌ 버그 있는 코드
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
df = load_msft(); df['Next_Open'] = df['Open'].shift(-1); df['Next_Close'] = df['Close'].shift(-1); df = df.dropna()
X = df[MSFT_FEATURES]; y = df[MSFT_TARGETS]
scaler = StandardScaler(); X = scaler.fit_transform(X)                                  # 버그1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)                # 버그2
model = Sequential([Input(shape=(5,)), Dense(64, activation='relu'),
                    Dense(1, activation='sigmoid')])                                    # 버그3
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])       # 버그4
model.fit(X_train, y_train, epochs=20)
print(model.predict(X_test)[:5])                                                        # 버그5

### C07. 다중 출력 회귀 응용: 역기구학 (pose 6 → joint 6), 출력별 원래 단위 MAE
**유형:** 회귀 / 다른 데이터에 적용 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** Lab2의 Forward Kinematics 코드(관절각→끝단 자세)를 **Inverse Kinematics**(끝단 `x,y,z,yaw,pitch,roll` → `q1_out~q6_out`)로 바꿔라.
64:16:20 분할, X·y 표준화, `Dense(128)-Dense(128)-Dense(6)`, EarlyStopping(patience=7), 학습곡선, 출력별 MAE(원래 단위) 출력.

**Search Keywords:** `df[[cols]].values` · `train_test_split` · `scaler_X` · `scaler_y` · `Dense(6)` · `EarlyStopping` · `inverse_transform` · `mean_absolute_error` · `enumerate(zip())`

**Source:** Lecture11_Lab2 p.44~49 (IRB2400 Forward Kinematics 코드, L11_05 Inverse 언급)

In [ ]:
# C07 연습 셀 — 빈칸(____)/TODO를 채워 실행
mount_drive()
df, FEAT, TARG = load_irb2400()     # IRB2400: FEAT=pose 6개, TARG=q1_out~q6_out
# TODO: Lab2 p.46~49 코드에서 X, y 열만 바꿔 적용

### C08. 시계열 직접 예측: 슬라이딩 윈도우 96→24, Last-Value/Seasonal 기준선, Residual MLP
**유형:** 회귀(시계열) / 통합 구현 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 최상

**문제:** ETTh1의 `OT`로
(1) 시간순 60/20/20 분할 후 train 평균·표준편차로 정규화 (2) `make_windows(series, 96, 24)` 구현
(3) Last Value, Seasonal Naive(24h) 기준선 MAE (4) Direct MLP: `Dense(128)-Dense(64)-Dense(24)`, shuffle=False
(5) Residual MLP: 목표 `r = y - x_t`, 예측 `ŷ = x_t + r̂` (6) MAE(°C) 비교표.

**Search Keywords:** `sliding window` · `make_windows` · `np.repeat` · `X[:, -24:]` · `shuffle=False` · `Dense(24)` · `residual` · `mean_absolute_error`

**Source:** Lecture11_Lab2 p.32~41 (ETTh1, 96→24 direct, baselines, residual forecasting)

In [ ]:
# C08 연습 셀 — 빈칸(____)/TODO를 채워 실행
mount_drive()
df = load_etth1()
ot = df['OT'].values.astype('float32')
def make_windows(series, in_len=96, out_len=24):
    ____

### C09. 목표값 로그 변환 (log1p / expm1) — 빈칸 채우기
**유형:** 회귀 / 빈칸 채우기 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** Ames 데이터에서 `SalePrice`를 `log(1+y)`로 변환해 학습하고, 예측을 달러로 복원해 MAE를 구하는 코드의 빈칸을 채워라.

**Search Keywords:** `np.log1p` · `np.expm1` · `flatten()` · `mean_absolute_error` · `kernel_initializer='he_normal'` · `Dropout(0.10)`

**Source:** Lecture11_Lab2 p.24~25 (log1p, expm1), p.13~14 (공통 설정, he_normal, val_mae EarlyStopping)

In [ ]:
# C09 연습 셀 — 빈칸(____)/TODO를 채워 실행
X_train_input, X_val_input, X_test_input, y_train, y_val, y_test = prepare_ames(load_ames())
y_train_log = np.____(y_train)
y_val_log = np.____(y_val)
# model_log 학습 ...
y_pred = np.____(model_log.predict(X_test_input, verbose=0).____())

## C — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### C01 정답 — MSFT 데이터 로드 → 결측치 제거 → 다음날 [Open, Close] 라벨 → 시간순 분할
**Expected Output:** y shape = (N, 2). train/val/test 기간이 겹치지 않고 시간순으로 이어짐

**Modification Point:** 예측 대상이 "다음날 시가"만이면 `TARGETS=["Next_Open"]` → 출력층 `Dense(1)`. n일 뒤 예측이면 `shift(-n)`. 입력에 과거 k일을 쓰면 `shift(1..k)` 열 추가 후 dropna.

In [ ]:
# ===== C01 정답 =====
mount_drive()
df = load_msft()                                   # 시험: df = pd.read_csv('주어진 MSFT 파일 경로')
print(df.columns.tolist(), df.shape)
df['Date'] = pd.to_datetime(df['Date'])            # (1)
df = df.sort_values('Date').set_index('Date')
# (2) 연습용 결측치 삽입 — 시험 데이터에 결측이 있으면 이 2줄은 삭제
df = df.copy()
df.iloc[[5, 50, 100], df.columns.get_loc('Close')] = np.nan
print('결측치:\n', df.isna().sum())
df = df.dropna(subset=MSFT_FEATURES).copy()
print('dropna 후:', df.shape)
# (3) 라벨 만들기: 다음날 값 = shift(-1)
FEATURES = ['Open', 'High', 'Low', 'Close', 'Volume']
TARGETS = ['Next_Open', 'Next_Close']
df['Next_Open'] = df['Open'].shift(-1)
df['Next_Close'] = df['Close'].shift(-1)
df = df.dropna(subset=TARGETS)                     # 마지막 행(다음날 없음) 제거
X = df.loc[:, FEATURES]
y = df.loc[:, TARGETS]                             # shape (N, 2)
# (4) 시간순 분할 (주가는 shuffle 금지 → 미래 정보 누수)
n = len(df); i1, i2 = int(n * 0.70), int(n * 0.85)
X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]
for name, a in [('train', X_train), ('val', X_val), ('test', X_test)]:
    print(f'{name:5s} {a.shape}  {a.index.min().date()} ~ {a.index.max().date()}')
# sklearn으로: train_test_split(X, y, test_size=0.15, shuffle=False)

### C02 정답 — 출력 2차원 회귀 MLP (Dense(2) linear, X·y 표준화, inverse_transform)
**Expected Output:** summary: Dense(64) 384, Dense(32) 2,080, Dense(2) 66 → 총 2,530 params. 출력별 MAE($) 2줄 + naive 비교

**Modification Point:** 출력 차원 k → `Dense(k)` + y 열 목록. 가격 범위가 test에서 크게 벗어나면 성능 저하 → 목표를 "변화량"(`Next_Close - Close`)으로 바꾸면 개선(Lab2 p.38 residual 아이디어).

In [ ]:
# ===== C02 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
tf.keras.utils.set_random_seed(42)
mount_drive()
X_train, X_val, X_test, y_train, y_val, y_test = make_msft_dataset()      # = C01
scaler_X, scaler_y = StandardScaler(), StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)        # train에만 fit
X_val_s, X_test_s = scaler_X.transform(X_val), scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train)        # y가 이미 2D (N, 2)
y_val_s = scaler_y.transform(y_val)
reg_model = Sequential([
    Input(shape=(X_train_s.shape[1],)),            # 5
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(2)                                       # 회귀 출력: 2차원, 활성화 없음(linear)
])
reg_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
reg_model.summary()
early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reg_history = reg_model.fit(X_train_s, y_train_s, validation_data=(X_val_s, y_val_s),
                            epochs=200, batch_size=32, callbacks=[early], verbose=0)
print('stopped at epoch', len(reg_history.history['loss']))
y_pred = scaler_y.inverse_transform(reg_model.predict(X_test_s, verbose=0))   # 달러 단위 복원
for i, name in enumerate(MSFT_TARGETS):
    print(f'{name}: MAE = {mean_absolute_error(y_test.iloc[:, i], y_pred[:, i]):.3f} $')
print('Naive(오늘 값 그대로) MAE:', mean_absolute_error(y_test['Next_Close'], X_test['Close']).round(3))

### C03 정답 — 사용자 정의 손실함수 (custom MSE · 가중 MSE · Huber) 구현과 적용
**Expected Output:** custom_mse [0.125 2.5 ] = keras mse [0.125 2.5 ]. Huber 샘플값 [0.0625, 1.0] → batch 평균 0.53125. 가중 손실은 Close MAE가 같거나 약간 감소(실행마다 차이)

**Modification Point:** 가중치 리스트 길이 = 출력 차원. MAE 기반이면 `tf.abs`. 손실 이름이 다르면 `custom_objects`의 key를 함수 `__name__`과 맞춤.

In [ ]:
# ===== C03 정답 =====
import numpy as np, tensorflow as tf
from tensorflow.keras.models import load_model

def custom_mse(y_true, y_pred):
    y_true = tf.cast(y_true, y_pred.dtype)               # float64 numpy 라벨 대비 (dtype 불일치 에러 방지)
    return tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)   # 샘플별 손실 (batch 평균은 Keras가)

def make_weighted_mse(weights):
    w = tf.constant(weights, dtype=tf.float32)
    def weighted_mse(y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        return tf.reduce_mean(w * tf.square(y_true - y_pred), axis=-1)
    return weighted_mse

def custom_huber(y_true, y_pred, delta=1.0):
    y_true = tf.cast(y_true, y_pred.dtype)
    err = y_true - y_pred
    abs_err = tf.abs(err)
    quad = 0.5 * tf.square(err)                         # |e| <= delta
    lin = delta * (abs_err - 0.5 * delta)               # |e| >  delta
    return tf.reduce_mean(tf.where(abs_err <= delta, quad, lin), axis=-1)

# (1) 검증: 내장 손실과 비교
yt = np.array([[1.0, 2.0], [3.0, 4.0]])
yp = np.array([[1.5, 2.0], [2.0, 6.0]], dtype='float32')
print('custom_mse :', custom_mse(yt, yp).numpy())
yt32 = yt.astype('float32')
print('keras mse  :', tf.keras.losses.mean_squared_error(yt32, yp).numpy())
print('custom_huber:', custom_huber(yt, yp).numpy(), '| keras:', tf.keras.losses.Huber(delta=1.0)(yt32, yp).numpy(), '(batch 평균)')

# (4) 적용: Next_Close에 2배 가중
weighted_mse = make_weighted_mse([1.0, 2.0])
res_w = train_msft_quick(epochs=100, loss=weighted_mse)
res_m = train_msft_quick(epochs=100, loss='mse')
from sklearn.metrics import mean_absolute_error
for tag, r in [('mse', res_m), ('weighted', res_w)]:
    maes = [mean_absolute_error(r['y_test'].iloc[:, i], r['y_pred'][:, i]) for i in range(2)]
    print(tag, 'MAE Open/Close =', np.round(maes, 3))

# (5) 저장/로드
os.makedirs('models', exist_ok=True)
try:
    res_w['model'].save('models/msft_weighted.keras')
    m1 = load_model('models/msft_weighted.keras', compile=False)                                  # 예측만
    m2 = load_model('models/msft_weighted.keras', custom_objects={'weighted_mse': weighted_mse})  # 재학습까지
    print('loaded OK')
except Exception as e:
    print('저장/로드 실패 시 → compile=False로 불러온 뒤 model.compile(loss=weighted_mse) 다시 호출:', type(e).__name__)

### C04 정답 — 모델 성능 시각화: 실제값 vs 예측값 (2개 출력 subplots + 산점도 y=x)
**Expected Output:** 선 그래프 2개(위: Open, 아래: Close), 산점도 2개(점이 y=x 근처면 좋음), 잔차 히스토그램

**Modification Point:** 출력이 k개면 `plt.subplots(k, 1)`. 일부 구간만: `y_test.iloc[:100]`. 학습곡선은 C05 `plot_history`.

In [ ]:
# ===== C04 정답 =====
res = train_msft_quick(epochs=100)          # C02 결과 대신 사용 (model, X_test, y_test, y_pred)
y_test, y_pred = res['y_test'], res['y_pred']
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)          # (1)
for i, name in enumerate(MSFT_TARGETS):
    axes[i].plot(y_test.index, y_test.iloc[:, i], label='Actual')
    axes[i].plot(y_test.index, y_pred[:, i], label='Predicted', alpha=0.8)
    axes[i].set_title(name); axes[i].set_ylabel('price ($)'); axes[i].legend(); axes[i].grid(True)
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))                      # (2)
for i, name in enumerate(MSFT_TARGETS):
    a, p = y_test.iloc[:, i].values, y_pred[:, i]
    axes[i].scatter(a, p, s=8, alpha=0.6)
    lim = [min(a.min(), p.min()), max(a.max(), p.max())]
    axes[i].plot(lim, lim, 'r--', label='y = x')
    axes[i].set_xlabel('Actual'); axes[i].set_ylabel('Predicted'); axes[i].set_title(name); axes[i].legend()
plt.tight_layout(); plt.show()
residual = y_test.values - y_pred                                      # (3)
plt.hist(residual, bins=40, label=MSFT_TARGETS, alpha=0.7); plt.xlabel('residual ($)'); plt.legend(); plt.show()

### C05 정답 — 학습곡선 (history.history → DataFrame, loss/val_loss, best epoch)
**Expected Output:** train/val 곡선 2개 그래프, best epoch 출력. EarlyStopping(restore_best_weights)이면 마지막 epoch = best + patience

**Modification Point:** 분류면 `accuracy`/`val_accuracy`. y축 범위 고정은 `plt.ylim([a, b])` (Lab1 p.39).

In [ ]:
# ===== C05 정답 =====
res = train_msft_quick(epochs=150)
history = res['history']
hist_df = pd.DataFrame(history.history)           # columns: loss, mae, val_loss, val_mae
print(hist_df.tail())
x_len = np.arange(1, len(hist_df) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_len, hist_df['loss'], 'o-', c='blue', markersize=2, label='Train loss')
axes[0].plot(x_len, hist_df['val_loss'], 'o-', c='red', markersize=2, label='Validation loss')
axes[1].plot(x_len, hist_df['mae'], label='Train MAE')
axes[1].plot(x_len, hist_df['val_mae'], label='Validation MAE')
for ax, yl in zip(axes, ['loss (MSE)', 'MAE']):
    ax.set_xlabel('epoch'); ax.set_ylabel(yl); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()
best = int(hist_df['val_loss'].idxmin()) + 1
print('best epoch:', best, '| min val_loss:', hist_df['val_loss'].min().round(5))
# 판단 기준: train loss는 계속 감소하는데 val loss가 최소 후 다시 증가 → 과적합 시작 (Lecture10 p.3)
# 재사용: plot_history(history, metric='mae')   ← 0번 SETUP 함수

### C06 정답 — [오류 수정] 회귀 코드에 섞인 분류 설정·누수·셔플 오류
**Expected Output:** 수정 후 출력 (5, 4) 배열: 실제 2열 + 예측 2열(달러 단위)

**Modification Point:** validation_split은 **마지막 20%**를 검증으로 씀(셔플 전) → 시계열에서도 시간순 유지됨.

In [ ]:
# ===== C06 정답 =====
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
df = load_msft()
df['Date'] = pd.to_datetime(df['Date']); df = df.sort_values('Date')
df['Next_Open'] = df['Open'].shift(-1); df['Next_Close'] = df['Close'].shift(-1); df = df.dropna()
X = df[MSFT_FEATURES].values; y = df[MSFT_TARGETS].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)   # 수정2: 시계열 → shuffle=False
sx, sy = StandardScaler(), StandardScaler()
X_train, X_test = sx.fit_transform(X_train), sx.transform(X_test)                          # 수정1: 분할 후 train에만 fit
y_train_s = sy.fit_transform(y_train)
model = Sequential([Input(shape=(5,)), Dense(64, activation='relu'),
                    Dense(2)])                                                             # 수정3: 출력 2차원, 활성화 없음
model.compile(optimizer='adam', loss='mse', metrics=['mae'])                               # 수정4: 회귀 손실/지표
model.fit(X_train, y_train_s, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
y_pred = sy.inverse_transform(model.predict(X_test, verbose=0))                            # 수정5: 원래 단위 복원
print(np.c_[y_test[:5], y_pred[:5]].round(2))   # [실제 Open, 실제 Close, 예측 Open, 예측 Close]

### C07 정답 — 다중 출력 회귀 응용: 역기구학 (pose 6 → joint 6), 출력별 원래 단위 MAE
**Expected Output:** 학습곡선 + 출력별 MAE 6줄(IRB2400, 단위 rad). 대체 데이터(linnerud)는 3줄

**Modification Point:** Forward ↔ Inverse는 `X`, `y` 열 목록만 교체. 출력 수가 바뀌면 `Dense(len(TARG))`가 자동 대응.

In [ ]:
# ===== C07 정답 =====
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
tf.keras.utils.set_random_seed(42)
mount_drive()
df, FEAT, TARG = load_irb2400()
X = df[FEAT].values                        # Inverse: pose → joint angles (Forward와 반대)
y = df[TARG].values
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.2, random_state=42)   # 64:16:20
scaler_X, scaler_y = StandardScaler(), StandardScaler()
X_train = scaler_X.fit_transform(X_train); X_val = scaler_X.transform(X_val); X_test = scaler_X.transform(X_test)
y_train = scaler_y.fit_transform(y_train); y_val = scaler_y.transform(y_val); y_test_scaled = scaler_y.transform(y_test)
model = Sequential([Input(shape=(len(FEAT),)), Dense(128, activation='relu'),
                    Dense(128, activation='relu'), Dense(len(TARG))])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
bs = 512 if len(X_train) > 5000 else 4
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50,
                    batch_size=bs, callbacks=[early_stopping], verbose=0)
plot_history(history, metric='mae')
test_mse, test_mae = model.evaluate(X_test, y_test_scaled, verbose=0)
print(f'Test (standardized) MSE {test_mse:.4f}  MAE {test_mae:.4f}')
y_pred = scaler_y.inverse_transform(model.predict(X_test, verbose=0))
for i, label in enumerate(TARG):
    print(f'MAE for {label:>8}: {mean_absolute_error(y_test[:, i], y_pred[:, i]):.4f}')

### C08 정답 — 시계열 직접 예측: 슬라이딩 윈도우 96→24, Last-Value/Seasonal 기준선, Residual MLP
**Expected Output:** 강의 결과(Lab2 p.40): Last Value 1.46, Seasonal 1.70, Basic MLP 1.61, Residual 1.36 °C 수준. 분할 비율/seed가 달라 값은 다를 수 있음

**Modification Point:** 입력/출력 길이 `in_len`, `out_len`과 `Dense(out_len)`을 함께 변경. 다변량이면 `series`를 (T, F)로 두고 `X.reshape(len(X), -1)`.

In [ ]:
# ===== C08 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
tf.keras.utils.set_random_seed(42)
mount_drive()
df = load_etth1()
ot = df['OT'].values.astype('float32')
n = len(ot); i1, i2 = int(n * 0.6), int(n * 0.8)                    # (1) 시간순 분할 (일반 구현: 비율 가정)
mu, sd = ot[:i1].mean(), ot[:i1].std()
z = (ot - mu) / sd                                                  # train 통계로 정규화

def make_windows(series, in_len=96, out_len=24):                    # (2)
    X, Y = [], []
    for t in range(in_len, len(series) - out_len + 1):
        X.append(series[t - in_len:t]); Y.append(series[t:t + out_len])
    return np.array(X), np.array(Y)

X_tr, Y_tr = make_windows(z[:i1]); X_va, Y_va = make_windows(z[i1:i2]); X_te, Y_te = make_windows(z[i2:])
print('windows:', X_tr.shape, Y_tr.shape, X_te.shape)
mae_c = lambda Y, P: np.mean(np.abs(Y - P)) * sd                     # 정규화 MAE → °C
results = {'Last Value': mae_c(Y_te, np.repeat(X_te[:, -1:], 24, axis=1)),          # (3)
           'Seasonal Naive (24h)': mae_c(Y_te, X_te[:, -24:])}

def build():
    m = Sequential([Input(shape=(96,)), Dense(128, activation='relu'), Dense(64, activation='relu'), Dense(24)])
    m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m
es = lambda: EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
direct = build()                                                    # (4)
direct.fit(X_tr, Y_tr, validation_data=(X_va, Y_va), epochs=50, batch_size=64, shuffle=False, callbacks=[es()], verbose=0)
results['Direct MLP'] = mae_c(Y_te, direct.predict(X_te, verbose=0))
resid = build()                                                     # (5) residual target
last = lambda X: X[:, -1:]
resid.fit(X_tr, Y_tr - last(X_tr), validation_data=(X_va, Y_va - last(X_va)),
          epochs=50, batch_size=64, shuffle=False, callbacks=[es()], verbose=0)
results['Last-Value Residual MLP'] = mae_c(Y_te, last(X_te) + resid.predict(X_te, verbose=0))
print(pd.Series(results, name='Test MAE (°C)').round(4))            # (6)

### C09 정답 — 목표값 로그 변환 (log1p / expm1) — 빈칸 채우기
**Expected Output:** Test MAE 약 $26,000 (Lab2 p.25: $26,047 — log 변환은 오히려 악화)

**Modification Point:** 표준화 방식으로 바꾸면 `StandardScaler` + `inverse_transform` (B05). 0 이하 값이 있으면 log1p도 주의.

In [ ]:
# ===== C09 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error
tf.keras.utils.set_random_seed(42)
mount_drive()
X_train_input, X_val_input, X_test_input, y_train, y_val, y_test = prepare_ames(load_ames())
y_train_log = np.log1p(y_train)                 # log(1 + y)
y_val_log = np.log1p(y_val)
input_dim = X_train_input.shape[1]
model_log = Sequential([Input(shape=(input_dim,)),
                        Dense(64, activation='relu', kernel_initializer='he_normal'), Dropout(0.10),
                        Dense(32, activation='relu', kernel_initializer='he_normal'), Dropout(0.10),
                        Dense(1)])
model_log.compile(loss='mse', optimizer=Adam(learning_rate=2e-4), metrics=['mae'])
early_stopping = EarlyStopping(monitor='val_mae', patience=30, mode='min', restore_best_weights=True)
model_log.fit(X_train_input, y_train_log, validation_data=(X_val_input, y_val_log),
              epochs=500, batch_size=32, callbacks=[early_stopping], verbose=0)
y_pred = np.expm1(model_log.predict(X_test_input, verbose=0).flatten())   # exp(ŷ) - 1 → 달러
print(f'Test MAE: ${mean_absolute_error(y_test, y_pred):,.0f}')

# D. 분류 (Lab1 · Lecture4)
- **D01** [하] Pima 이진 분류 MLP + 파라미터 수 계산
- **D02** [중] sigmoid 이진 분류 → softmax 2클래스 변환 (to_categorical)
- **D03** [중] Iris 다중분류: get_dummies vs LabelEncoder+sparse — 빈칸 채우기
- **D04** [하] [오류 수정] 라벨 형식과 손실함수 불일치
- **D05** [상] Wine 분류: stratify 분할 + validation_split + EarlyStopping + ModelCheckpoint + load_model
- **D06** [중] 혼동행렬로 Accuracy·Precision·Recall·F1 직접 계산 + 불균형 데이터 함정
- **D07** [하] 다중 라벨 분류 (sigmoid + binary_crossentropy + threshold 0.5) — 빈칸 채우기

## D — 문제

### D01. Pima 이진 분류 MLP + 파라미터 수 계산
**유형:** 분류 / 출력 예측 + 코드 작성 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 하

**문제:** (1) Lab1 구조 `8 → Dense(12) → Dense(8) → Dense(1, sigmoid)`의 층별 파라미터 수와 총합을 손으로 계산
(2) 은닉층을 `Dense(16) → Dense(8)`로 바꿨을 때 총 파라미터 수 (3) 모델을 구성·학습(epochs=100, batch_size=10)하고 `summary()`로 확인
(4) 한 epoch당 파라미터 갱신 횟수는?

**Search Keywords:** `Sequential` · `Input(shape=(8,))` · `Dense(1, activation='sigmoid')` · `binary_crossentropy` · `model.summary` · `batch_size` · `df.iloc[:, 0:8]`

**Source:** Lecture6_Lab1 p.13~16 (모델, 파라미터 221, compile/fit, 77 updates)

In [ ]:
# D01 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
mount_drive(); df = load_pima()
X = df.iloc[:, ____]; y = df.iloc[:, ____]
# TODO

### D02. sigmoid 이진 분류 → softmax 2클래스 변환 (to_categorical)
**유형:** 분류 / 모델 수정 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** D01 모델을 softmax 출력으로 바꿔라. (1) 라벨 원-핫 (2) 출력층 `Dense(2, softmax)` (3) 손실 변경
(4) 예측 확률에서 `argmax`로 클래스 결정 (5) sigmoid 버전과 정확도 비교(validation_split=0.2).

**Search Keywords:** `to_categorical` · `Dense(2, activation='softmax')` · `categorical_crossentropy` · `argmax(axis=1)` · `validation_split`

**Source:** Lecture6_Lab1 p.18 (to_categorical, softmax 2), Lecture4 p.21 (softmax regression code)

In [ ]:
# D02 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.utils import to_categorical
# TODO

### D03. Iris 다중분류: get_dummies vs LabelEncoder+sparse — 빈칸 채우기
**유형:** 분류 / 빈칸 채우기 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** Iris 3클래스 분류에서 (A) 원-핫 방식과 (B) 정수 라벨 방식의 빈칸을 채우고, 총 파라미터 수(4→12→8→3)를 계산하라.

**Search Keywords:** `pd.get_dummies(dtype=int)` · `LabelEncoder` · `fit_transform` · `categorical_crossentropy` · `sparse_categorical_crossentropy` · `Dense(3, activation='softmax')` · `sns.pairplot`

**Source:** Lecture6_Lab1 p.22~28 (iris, get_dummies, 191 params, LabelEncoder, sparse_categorical_crossentropy)

In [ ]:
# D03 연습 셀 — 빈칸(____)/TODO를 채워 실행
mount_drive(); df = load_iris_df()
X = df.iloc[:, 0:4]; y = df.iloc[:, 4]
# (A)
y_onehot = pd.____(y, dtype=int)
model.add(Dense(3, activation='____'))
model.compile(loss='____', optimizer='adam', metrics=['accuracy'])
# (B)
le = ____(); y_int = le.____(y)
model.compile(loss='____', optimizer='adam', metrics=['accuracy'])

### D04. [오류 수정] 라벨 형식과 손실함수 불일치
**유형:** 분류 / 오류 수정 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 하

**문제:** 다음 3개 코드 조각은 각각 오류(또는 잘못된 학습)가 난다. 원인과 수정 방법을 쓰라.

**Search Keywords:** `sparse_categorical_crossentropy` · `categorical_crossentropy` · `to_categorical` · `binary_crossentropy` · `softmax` · `sigmoid`

**Source:** Lecture6_Lab1 p.18, p.28, p.29; Lecture9 p.12~13 (출력층)

In [ ]:
# D04 연습 셀 — 빈칸(____)/TODO를 채워 실행
# (a) 정수 라벨 y_int(0,1,2) + Dense(3, softmax) + loss='categorical_crossentropy'
# (b) 3클래스인데 Dense(1, activation='sigmoid') + loss='binary_crossentropy'
# (c) 다중 라벨(multi-hot) [1,0,1,0,0] 인데 Dense(5, activation='softmax') + categorical_crossentropy

### D05. Wine 분류: stratify 분할 + validation_split + EarlyStopping + ModelCheckpoint + load_model
**유형:** 분류 / 과적합 방지 + 모델 저장 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** wine.csv(header 없음, 0~11열 특성, 12열 레이블)로
(1) test 20%를 `stratify=y`로 분리 (2) 나머지의 25%를 `validation_split`으로 사용(최종 6:2:2)
(3) `EarlyStopping(val_loss, patience=20)` + `ModelCheckpoint(save_best_only=True)` (4) best 모델을 불러와 test 평가.

**Search Keywords:** `header=None` · `train_test_split` · `stratify=y` · `validation_split=0.25` · `EarlyStopping` · `patience` · `ModelCheckpoint` · `save_best_only=True` · `.keras` · `load_model` · `model.evaluate`

**Source:** Lecture6_Lab1 p.32~43 (wine, stratify, validation_split=0.25, ModelCheckpoint, EarlyStopping, load_model)

In [ ]:
# D05 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
mount_drive(); df = load_wine_rw()
# TODO

### D06. 혼동행렬로 Accuracy·Precision·Recall·F1 직접 계산 + 불균형 데이터 함정
**유형:** 분류 평가 / 코드 작성 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** (1) 주어진 `y_true, y_pred`에서 TP·FP·FN·TN을 NumPy로 직접 세고 4개 지표 계산 (2) sklearn 결과와 비교
(3) 10,000명 중 5명만 양성인데 모두 음성으로 예측하는 모델의 accuracy/precision/recall (Lecture4 p.27 재현).

**Search Keywords:** `confusion_matrix` · `ravel()` · `accuracy_score` · `precision_score` · `recall_score` · `f1_score` · `classification_report` · `zero_division=0`

**Source:** Lecture4 p.24~27 (confusion matrix, metrics, imbalanced example)

In [ ]:
# D06 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0, 0, 0])
# TODO

### D07. 다중 라벨 분류 (sigmoid + binary_crossentropy + threshold 0.5) — 빈칸 채우기
**유형:** 분류 / 빈칸 채우기 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 하

**문제:** 한 샘플이 여러 클래스에 동시에 속하는 5클래스 다중 라벨 문제의 빈칸을 채우고, 예측 확률을 0.5 기준으로 multi-hot 벡터로 변환하라.

**Search Keywords:** `multi-hot` · `Dense(5, activation='sigmoid')` · `binary_crossentropy` · `(pred >= 0.5).astype(int)`

**Source:** Lecture6_Lab1 p.29 (Multi-Label Classification)

In [ ]:
# D07 연습 셀 — 빈칸(____)/TODO를 채워 실행
model = Sequential([Input(shape=(10,)), Dense(8, activation='relu'), Dense(5, activation='____')])
model.compile(optimizer='adam', loss='____', metrics=['accuracy'])
pred = model.predict(X_new)
pred_labels = (pred ____ 0.5).astype(____)

## D — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### D01 정답 — Pima 이진 분류 MLP + 파라미터 수 계산
**Expected Output:** (1) 108 + 104 + 9 = 221 (2) 8·16+16=144, 16·8+8=136, 9 → 289 (4) ceil(768/10) = 77

**Modification Point:** 파라미터 = (입력 수 × 뉴런 수) + 뉴런 수(bias). 입력 특성 수가 바뀌면 `Input(shape=(k,))`와 `iloc[:, 0:k]`.

In [ ]:
# ===== D01 정답 =====
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
mount_drive()
df = load_pima()
X = df.iloc[:, 0:8]          # column 0~7
y = df.iloc[:, 8]            # column 8 (diabetes)
model = Sequential()
model.add(Input(shape=(8,)))
model.add(Dense(12, activation='relu', name='Dense_1'))    # 8*12+12 = 108
model.add(Dense(8, activation='relu', name='Dense_2'))     # 12*8+8 = 104
model.add(Dense(1, activation='sigmoid', name='Dense_3'))  # 8*1+1  = 9   → 총 221
model.summary()
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, batch_size=10, verbose=0)
print('updates/epoch =', int(np.ceil(len(X) / 10)))          # 768/10 → 77
prob = model.predict(X.iloc[:5], verbose=0).ravel()
print('P(diabetes=1):', prob.round(3), '→ class', (prob >= 0.5).astype(int))

### D02 정답 — sigmoid 이진 분류 → softmax 2클래스 변환 (to_categorical)
**Expected Output:** softmax 예측 각 행 합 = 1, argmax로 0/1. 두 방식 정확도는 비슷(수학적으로 동등한 모델)

**Modification Point:** 정수 라벨 그대로 쓰려면 `loss="sparse_categorical_crossentropy"` (원-핫 불필요).

In [ ]:
# ===== D02 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
tf.keras.utils.set_random_seed(42)
mount_drive(); df = load_pima()
X = df.iloc[:, 0:8].values; y = df.iloc[:, 8].values
def build(out_units, out_act):
    return Sequential([Input(shape=(8,)), Dense(12, activation='relu'), Dense(8, activation='relu'),
                       Dense(out_units, activation=out_act)])
m_sig = build(1, 'sigmoid')
m_sig.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
h1 = m_sig.fit(X, y, epochs=100, batch_size=10, validation_split=0.2, verbose=0)
y_cat = to_categorical(y)                            # 0 → [1,0], 1 → [0,1]
m_soft = build(2, 'softmax')
m_soft.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
h2 = m_soft.fit(X, y_cat, epochs=100, batch_size=10, validation_split=0.2, verbose=0)
p = m_soft.predict(X[:5], verbose=0)
print(p.round(3), '→', p.argmax(axis=1))            # 확률 합 = 1
print('val acc sigmoid:', round(h1.history['val_accuracy'][-1], 4), '| softmax:', round(h2.history['val_accuracy'][-1], 4))

### D03 정답 — Iris 다중분류: get_dummies vs LabelEncoder+sparse — 빈칸 채우기
**Expected Output:** 총 파라미터 191 (Lab1 p.25). 두 방식 정확도 ≈ 0.95~0.98

**Modification Point:** 클래스 수 C → `Dense(C, softmax)`. 원-핫 라벨 + sparse 손실(또는 반대)은 shape 오류.

In [ ]:
# ===== D03 정답 =====
import tensorflow as tf, seaborn as sns
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
tf.keras.utils.set_random_seed(42)
mount_drive(); df = load_iris_df()
X = df.iloc[:, 0:4].values.astype('float32'); y = df.iloc[:, 4]
# sns.pairplot(df, hue=df.columns[4]); plt.show()     # Lab1 p.23
def build():
    return Sequential([Input(shape=(4,)), Dense(12, activation='relu'), Dense(8, activation='relu'),
                       Dense(3, activation='softmax')])   # 48+12 / 96+8 / 24+3 = 191
# (A) one-hot
y_onehot = pd.get_dummies(y, dtype=int)               # setosa→[1,0,0] ...
mA = build(); mA.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
mA.fit(X, y_onehot.values, epochs=100, batch_size=10, verbose=0)
# (B) integer labels
le = LabelEncoder(); y_int = le.fit_transform(y)      # ['setosa','versicolor','virginica'] → [0,1,2]
mB = build(); mB.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
mB.fit(X, y_int, epochs=100, batch_size=10, verbose=0)
mB.summary()
print('classes:', list(le.classes_))
print('A acc:', mA.evaluate(X, y_onehot.values, verbose=0)[1].round(3), '| B acc:', round(mB.evaluate(X, y_int, verbose=0)[1], 3))
print('예측 클래스 이름:', le.inverse_transform(mB.predict(X[:3], verbose=0).argmax(axis=1)))

### D04 정답 — [오류 수정] 라벨 형식과 손실함수 불일치
**Expected Output:** (a) sparse_categorical_crossentropy 또는 to_categorical  (b) Dense(3, softmax)  (c) Dense(5, sigmoid) + BCE

**Modification Point:** 정답 표: 이진=Dense(1,sigmoid)+BCE / 다중=Dense(C,softmax)+CCE(원-핫) 또는 SCCE(정수) / 다중라벨=Dense(C,sigmoid)+BCE / 회귀=Dense(k)+MSE

In [ ]:
# ===== D04 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
X = np.random.rand(30, 4).astype('float32'); y_int = np.random.randint(0, 3, 30)   # 구조 확인용 임의 데이터
# (a) 원인: categorical_crossentropy는 원-핫(N,3)을 기대 → 정수(N,) 라벨이면 shape 오류
m = Sequential([Input(shape=(4,)), Dense(8, activation='relu'), Dense(3, activation='softmax')])
m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')      # 수정1
m.fit(X, y_int, epochs=1, verbose=0)
m.compile(loss='categorical_crossentropy', optimizer='adam')             # 수정2: 라벨을 원-핫으로
m.fit(X, to_categorical(y_int, num_classes=3), epochs=1, verbose=0)
# (b) 원인: 출력 1개로는 3클래스 확률을 못 냄 → Dense(3, softmax) + (sparse_)categorical_crossentropy
# (c) 원인: softmax는 확률 합이 1 → 여러 클래스 동시 정답 불가 → Dense(5, sigmoid) + binary_crossentropy
print('(a) 수정 후 학습 OK')

### D05 정답 — Wine 분류: stratify 분할 + validation_split + EarlyStopping + ModelCheckpoint + load_model
**Expected Output:** 수백 epoch에서 조기 종료. Test accuracy ≈ 0.99 (Lab1 p.43: epoch 389 종료, 0.99)

**Modification Point:** `restore_best_weights=True`를 쓰면 checkpoint 없이도 best 가중치 복원. 매 epoch 저장은 `"{epoch:02d}-{val_accuracy:.4f}.keras"` (Lab1 p.35).

In [ ]:
# ===== D05 정답 =====
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
tf.keras.utils.set_random_seed(42)
mount_drive()
df = load_wine_rw()
X = df.iloc[:, 0:12]; y = df.iloc[:, 12]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)
model = Sequential([Input(shape=(12,)), Dense(30, activation='relu'), Dense(12, activation='relu'),
                    Dense(8, activation='relu'), Dense(1, activation='sigmoid')])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
save_dir = MODEL_DIR if os.path.isdir('/content/drive/MyDrive') else 'models'   # Drive 없으면 로컬 폴더
os.makedirs(save_dir, exist_ok=True)
modelpath = os.path.join(save_dir, 'best_wine_model.keras')       # Keras 3: .keras 확장자 필수
checkpointer = ModelCheckpoint(filepath=modelpath, monitor='val_loss', save_best_only=True)
early_stopping = EarlyStopping(monitor='val_loss', patience=20)
history = model.fit(X_train, y_train, epochs=2000, batch_size=500, validation_split=0.25,
                    callbacks=[early_stopping, checkpointer], verbose=0)
print('stopped at epoch', len(history.history['loss']))
best_model = load_model(modelpath)
score = best_model.evaluate(X_test, y_test, verbose=0)
print('Test loss:', round(score[0], 4), 'Test accuracy:', round(score[1], 4))

### D06 정답 — 혼동행렬로 Accuracy·Precision·Recall·F1 직접 계산 + 불균형 데이터 함정
**Expected Output:** TP=3 FP=1 FN=2 TN=4 → acc 0.70, prec 0.75, rec 0.60, f1 0.667. 불균형: acc 0.9995, precision 0(측정 불가), recall 0

**Modification Point:** 주의: 강의 표(Lecture4 p.24)는 행=예측, 열=실제 / sklearn은 행=실제, 열=예측. 양성 클래스가 다르면 `pos_label=`.

In [ ]:
# ===== D06 정답 =====
import numpy as np
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0, 0, 0])
TP = np.sum((y_true == 1) & (y_pred == 1)); TN = np.sum((y_true == 0) & (y_pred == 0))
FP = np.sum((y_true == 0) & (y_pred == 1)); FN = np.sum((y_true == 1) & (y_pred == 0))
acc = (TP + TN) / (TP + TN + FP + FN); prec = TP / (TP + FP); rec = TP / (TP + FN)
f1 = 2 * prec * rec / (prec + rec)
print(f'TP={TP} FP={FP} FN={FN} TN={TN} | acc={acc:.2f} prec={prec:.2f} rec={rec:.2f} f1={f1:.3f}')
cm = confusion_matrix(y_true, y_pred)          # sklearn: 행=실제, 열=예측 → [[TN, FP], [FN, TP]]
print(cm); tn, fp, fn, tp = cm.ravel()
print(accuracy_score(y_true, y_pred), precision_score(y_true, y_pred), recall_score(y_true, y_pred), round(f1_score(y_true, y_pred), 3))
print(classification_report(y_true, y_pred, digits=3))
# (3) 불균형: 양성 5명 / 10,000명, 항상 음성 예측
yt = np.zeros(10000, dtype=int); yt[:5] = 1; yp = np.zeros(10000, dtype=int)
print('acc', accuracy_score(yt, yp), '| prec', precision_score(yt, yp, zero_division=0), '| recall', recall_score(yt, yp))

### D07 정답 — 다중 라벨 분류 (sigmoid + binary_crossentropy + threshold 0.5) — 빈칸 채우기
**Expected Output:** 확률 행렬(행 합 ≠ 1)과 0/1 multi-hot 행렬

**Modification Point:** 임계값을 클래스별로 다르게: `(pred >= np.array([0.5, 0.3, ...]))`.

In [ ]:
# ===== D07 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
tf.keras.utils.set_random_seed(0)
X = np.random.rand(200, 10).astype('float32')                    # 구조 확인용 임의 입력 (실제 데이터 아님)
Y = (np.random.rand(200, 5) > 0.6).astype('float32')             # multi-hot 라벨 예: [1,0,1,0,0]
model = Sequential([Input(shape=(10,)), Dense(8, activation='relu'),
                    Dense(5, activation='sigmoid')])             # 클래스별 독립 확률
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X, Y, epochs=5, verbose=0)
pred = model.predict(X[:3], verbose=0)
pred_labels = (pred >= 0.5).astype(int)
print(pred.round(2)); print(pred_labels)                          # 행 합이 1이 아닐 수 있음

# E. Fashion-MNIST · CNN · 증강 · Weight decay · 과적합 방지 · Confusion matrix
- **E01** [하] Fashion-MNIST 불러오기 + 0~1 정규화 + 채널 차원 추가
- **E02** [중] 훈련/검증 데이터 분리 (stratify) — test는 손대지 않기
- **E03** [하] 훈련 이미지 10개 2×5 그리드 시각화 (클래스 이름 제목)
- **E04** [상] 훈련 데이터에만 실시간 데이터 증강 2가지 이상 (ImageDataGenerator.flow)
- **E05** [상] Conv2D 기반 CNN 설계·학습·평가 + 파라미터 수
- **E06** [상] Weight decay 적용: (A) kernel_regularizer=l2 (B) AdamW
- **E07** [중] 과적합 방지: Dropout + EarlyStopping(restore_best_weights) + ModelCheckpoint
- **E08** [중] Confusion matrix 출력 (ConfusionMatrixDisplay, 정규화, 가장 헷갈린 쌍)
- **E09** [상] [오류 수정] CNN 파이프라인의 7가지 실수
- **E10** [상] CNN에 BatchNormalization 추가 (Conv → BN → ReLU)
- **E11** [중] 증강 결과 시각화: 원본 1장 + 증강 9장 (2×5)
- **E12** [최상] [대체 구현] Keras 전처리 레이어로 모델 내부 증강 (RandomFlip/Rotation/Translation/Zoom)

## E — 문제

### E01. Fashion-MNIST 불러오기 + 0~1 정규화 + 채널 차원 추가
**유형:** CNN 데이터 전처리 / 코드 작성 · **출제 근거:** 족보 직접 등장 · **난이도:** 하

**문제:** `tensorflow.keras.datasets.fashion_mnist`를 불러와 (1) shape·dtype·픽셀 범위 출력 (2) `float32`로 바꿔 255로 나눠 0~1 정규화
(3) Conv2D 입력을 위해 `(N, 28, 28, 1)`로 reshape (4) 정규화 후 min/max와 라벨 값 범위(0~9) 확인.

**Search Keywords:** `fashion_mnist.load_data` · `astype('float32')` · `/ 255.0` · `reshape(-1, 28, 28, 1)` · `np.expand_dims` · `x.min() x.max()` · `np.unique`

**Source:** 족보 "Fashion-MNIST 불러오기 및 0~1 정규화"; Lecture10 p.29, Lecture9 p.30 (mnist reshape /255)

In [ ]:
# E01 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.datasets import fashion_mnist
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()
# TODO

### E02. 훈련/검증 데이터 분리 (stratify) — test는 손대지 않기
**유형:** CNN 데이터 전처리 / 코드 작성 · **출제 근거:** 족보 직접 등장 · **난이도:** 중

**문제:** E01의 훈련 데이터 60,000장을 train 48,000 / validation 12,000으로 분리하라(stratify, random_state=42).
(1) 각 shape (2) 클래스별 개수(`np.bincount`)가 균등한지 확인 (3) 슬라이싱 방식 분리와의 차이를 주석으로.

**Search Keywords:** `train_test_split` · `test_size=0.2` · `stratify=y` · `random_state` · `np.bincount` · `x[:-12000]` · `validation_data`

**Source:** 족보 "훈련 데이터와 검증 데이터 분리"; Lecture10 p.2 (train/val/test 역할), Lecture6_Lab1 p.33 (stratify)

In [ ]:
# E02 연습 셀 — 빈칸(____)/TODO를 채워 실행
from sklearn.model_selection import train_test_split
# TODO: x_train, x_val, y_train, y_val = ...

### E03. 훈련 이미지 10개 2×5 그리드 시각화 (클래스 이름 제목)
**유형:** 시각화 / 코드 작성 · **출제 근거:** 족보 직접 등장 · **난이도:** 하

**문제:** 훈련 데이터 앞 10장을 `plt.subplots(2, 5)`로 그리고 제목에 클래스 이름을 표시하라(축 숨김, gray).
변형: 클래스 0~9에서 한 장씩 골라 2×5로 그리기.

**Search Keywords:** `plt.subplots(2, 5)` · `axes.flat` · `imshow` · `cmap='gray'` · `squeeze()` · `set_title` · `axis('off')` · `tight_layout` · `np.where(y == c)[0][0]`

**Source:** 족보 "훈련 이미지 10개를 2×5 그리드로 시각화"

In [ ]:
# E03 연습 셀 — 빈칸(____)/TODO를 채워 실행
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
fig, axes = plt.subplots(____, ____, figsize=(12, 5))
for i, ax in enumerate(axes.____):
    ax.imshow(x_train[i].____(), cmap='gray')
    ax.set_title(CLASS_NAMES[y_train[i]]); ax.axis('off')
plt.tight_layout(); plt.show()

### E04. 훈련 데이터에만 실시간 데이터 증강 2가지 이상 (ImageDataGenerator.flow)
**유형:** CNN / 데이터 증강 · **출제 근거:** 족보 직접 등장 · **난이도:** 상

**문제:** (1) `ImageDataGenerator`로 회전·이동·확대·좌우반전 중 **2가지 이상** 적용한 train 전용 생성기 생성
(2) `flow(x_train, y_train, batch_size=64)`로 배치를 만들고 배치 shape·값 범위 확인 (3) 검증/테스트는 **증강하지 않고** 그대로 사용하는 fit 코드 작성
(4) 왜 검증 데이터는 증강하면 안 되는지 주석으로.

**Search Keywords:** `ImageDataGenerator` · `rotation_range` · `width_shift_range` · `height_shift_range` · `zoom_range` · `horizontal_flip` · `datagen.flow` · `next()` · `validation_data=(x_val, y_val)` · `real-time augmentation`

**Source:** 족보 "훈련 데이터에만 실시간 데이터 증강 2가지 이상"; Lecture10 p.28~29 (augmentation 종류, ImageDataGenerator 코드, 실시간 방식 설명)

In [ ]:
# E04 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.preprocessing.image import ImageDataGenerator
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
# TODO

### E05. Conv2D 기반 CNN 설계·학습·평가 + 파라미터 수
**유형:** CNN / 모델 구현 · **출제 근거:** 족보 직접 등장 · **난이도:** 상

**문제:** `Input(28,28,1) → Conv2D(32,3×3,relu,same) → MaxPool(2) → Conv2D(64,3×3,relu,same) → MaxPool(2) → Flatten → Dense(128,relu) → Dense(10,softmax)`
(1) 모델 구성·summary, 각 층 파라미터 수 손계산 (2) 증강 생성기(E04)로 학습, 검증은 원본 (3) test 정확도 출력.

**Search Keywords:** `Conv2D` · `MaxPooling2D` · `Flatten` · `padding='same'` · `Dense(10, activation='softmax')` · `sparse_categorical_crossentropy` · `model.summary` · `model.fit(train_gen)` · `model.evaluate`

**Source:** 족보 "Conv2D 기반 CNN 설계"; Lecture10 p.29 (Conv2D-MaxPooling2D-Flatten-Dense 코드)

In [ ]:
# E05 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
# TODO

### E06. Weight decay 적용: (A) kernel_regularizer=l2 (B) AdamW
**유형:** CNN / 정규화(weight decay) · **출제 근거:** 족보 직접 등장 · **난이도:** 상

**문제:** E05 CNN에 weight decay를 두 가지 방식으로 적용하라.
(A) 모든 Conv2D·Dense에 `kernel_regularizer=l2(1e-4)` + Adam (B) 구조는 그대로, `AdamW(learning_rate=1e-3, weight_decay=1e-4)`
(1) (A)에서 `model.losses` 개수 확인 (2) 두 모델 각 3 epoch 학습 후 val accuracy 비교 (3) 두 방식의 차이를 주석으로.

**Search Keywords:** `kernel_regularizer` · `l2(1e-4)` · `tensorflow.keras.regularizers` · `AdamW` · `weight_decay` · `model.losses` · `decoupled weight decay`

**Source:** 족보 "Weight decay 적용"; Lecture10 p.19~22 (weight decay, AdamW 코드, kernel_regularizer=l2 코드), Lecture8 p.13

In [ ]:
# E06 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW
# TODO

### E07. 과적합 방지: Dropout + EarlyStopping(restore_best_weights) + ModelCheckpoint
**유형:** CNN / 과적합 방지 · **출제 근거:** 족보 직접 등장 · **난이도:** 중

**문제:** E05 CNN에 (1) Flatten 뒤·Dense 뒤 `Dropout` 추가 (2) `EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)`
(3) `ModelCheckpoint(save_best_only=True)` (4) 학습 후 train-val 정확도 격차를 출력하고, 사용한 과적합 방지 기법 목록을 주석으로 정리.

**Search Keywords:** `Dropout(0.5)` · `EarlyStopping` · `monitor='val_loss'` · `patience` · `restore_best_weights=True` · `ModelCheckpoint` · `save_best_only` · `callbacks=[...]`

**Source:** 족보 "과적합 방지 기법 적용"; Lecture10 p.3, p.7, p.24~26 (Dropout 코드); Lecture6_Lab1 p.41~42

In [ ]:
# E07 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# TODO

### E08. Confusion matrix 출력 (ConfusionMatrixDisplay, 정규화, 가장 헷갈린 쌍)
**유형:** 모델 평가 / 시각화 · **출제 근거:** 족보 직접 등장 · **난이도:** 중

**문제:** 학습된 CNN으로 test 예측을 만들어
(1) `predict` 확률 → `argmax(axis=1)`로 클래스 (2) `confusion_matrix` 출력 및 `ConfusionMatrixDisplay`로 클래스 이름 표시
(3) 행 기준 정규화(`normalize='true'`) heatmap (4) 대각선 제외 가장 많이 헷갈린 (실제, 예측) 쌍 (5) `classification_report`.

**Search Keywords:** `model.predict` · `argmax(axis=1)` · `confusion_matrix` · `ConfusionMatrixDisplay` · `display_labels` · `normalize='true'` · `np.fill_diagonal` · `unravel_index` · `classification_report` · `target_names`

**Source:** 족보 "Confusion matrix 출력"; Lecture4 p.24~26 (confusion matrix, precision/recall/F1)

In [ ]:
# E08 연습 셀 — 빈칸(____)/TODO를 채워 실행
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
# cnn_model이 없으면: cnn_model, _, (x_test, y_test) = train_cnn_quick()
# TODO

### E09. [오류 수정] CNN 파이프라인의 7가지 실수
**유형:** CNN / 오류 수정 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** 다음 코드의 오류·잘못된 관행 **7개**를 찾아 고쳐라 (실행 에러 + 결과를 왜곡하는 실수 포함).

**Search Keywords:** `reshape(-1, 28, 28, 1)` · `rescale` · `sparse_categorical_crossentropy` · `softmax` · `validation_split generator` · `validation_data` · `evaluate(x_test, y_test)`

**Source:** Lecture10 p.29 (올바른 코드), Lecture6_Lab1 p.28 (sparse), 족보 증강·CNN 항목

In [ ]:
# E09 연습 셀 — 빈칸(____)/TODO를 채워 실행
# ❌ 버그 있는 코드
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train / 255.0                                                  # ①
datagen = ImageDataGenerator(rescale=1/255., rotation_range=10, zoom_range=0.1)   # ②
model = Sequential([Input(shape=(28, 28)),                                 # ③
                    Conv2D(32, (3, 3), activation='relu'), MaxPooling2D((2, 2)),
                    Flatten(), Dense(64, activation='relu'),
                    Dense(10, activation='relu')])                         # ④
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])   # ⑤
model.fit(datagen.flow(x_train, y_train), validation_split=0.2, epochs=5)  # ⑥
model.evaluate(datagen.flow(x_test, y_test))                               # ⑦

### E10. CNN에 BatchNormalization 추가 (Conv → BN → ReLU)
**유형:** CNN / 모델 확장 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** E05 CNN의 각 Conv2D/Dense 뒤에 `BatchNormalization`을 **활성화 함수 앞에** 넣어라.
(1) `Conv2D(..., use_bias=False)` → `BatchNormalization()` → `Activation('relu')` (2) BN 유무 모델을 각 3 epoch 학습해 val accuracy 비교
(3) BN의 학습/추론 시 동작 차이를 주석으로.

**Search Keywords:** `BatchNormalization` · `Activation('relu')` · `use_bias=False` · `running mean` · `training vs inference`

**Source:** Lecture9 p.23~30 (BN 수식, test-time, BN 코드), Lecture11_Lab2 p.16~17 (BN 결과)

In [ ]:
# E10 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.layers import BatchNormalization, Activation
# TODO

### E11. 증강 결과 시각화: 원본 1장 + 증강 9장 (2×5)
**유형:** 시각화 / 데이터 증강 확인 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** E04의 증강 설정이 제대로 적용되는지 확인하라. 훈련 이미지 1장에 대해 원본 + 증강 9장을 2×5 그리드로 표시하고, 증강 이미지의 값 범위가 0~1인지 확인.

**Search Keywords:** `datagen.flow(img, batch_size=1)` · `next(it)` · `random_transform` · `plt.subplots(2, 5)` · `imshow`

**Source:** Lecture10 p.28~29 (augmentation 종류와 코드)

In [ ]:
# E11 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.preprocessing.image import ImageDataGenerator
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
# TODO

### E12. [대체 구현] Keras 전처리 레이어로 모델 내부 증강 (RandomFlip/Rotation/Translation/Zoom)
**유형:** CNN / 데이터 증강 (대체 구현) · **출제 근거:** 강의자료 기반 예상 (강의자료 외 대체 구현 포함) · **난이도:** 최상

**문제:** `ImageDataGenerator` 대신 Keras 전처리 레이어로 **모델 안에서** 실시간 증강하라.
(1) `Sequential([RandomFlip('horizontal'), RandomRotation(0.05), RandomTranslation(0.1, 0.1), RandomZoom(0.1)])` 블록 생성
(2) `training=False`일 때 입력이 그대로 나오는지(= 검증·테스트는 자동으로 증강 안 됨) 확인 (3) CNN 앞단에 붙여 학습·평가.
※ 강의자료(Lecture10 p.29)는 ImageDataGenerator 방식 — 이 문제는 일반적인 대체 구현임.

**Search Keywords:** `RandomFlip` · `RandomRotation` · `RandomTranslation` · `RandomZoom` · `training=False` · `keras.layers preprocessing`

**Source:** Lecture10 p.28~29 개념의 대체 구현 (Keras 공식 preprocessing layers). 강의자료 코드 아님

In [ ]:
# E12 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras import layers
# TODO

## E — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### E01 정답 — Fashion-MNIST 불러오기 + 0~1 정규화 + 채널 차원 추가
**Expected Output:** (60000, 28, 28) uint8 0 255 → (60000, 28, 28, 1) float32 0.0 1.0, test (10000, 28, 28)

**Modification Point:** Dense(MLP) 입력이면 `reshape(-1, 784)` (Lecture9 p.30). 표준화 요구 시 train 평균/표준편차로 `(x - mean) / std`.

In [ ]:
# ===== E01 정답 =====
from tensorflow.keras.datasets import fashion_mnist
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()
print(x_train_full.shape, x_train_full.dtype, x_train_full.min(), x_train_full.max())   # (60000, 28, 28) uint8 0 255
print(x_test.shape, y_train_full.shape, np.unique(y_train_full))                        # 라벨 0~9 (정수)
x_train_full = x_train_full.reshape(-1, 28, 28, 1).astype('float32') / 255.0             # 이미지에만 적용 (라벨 X)
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0                         # test도 같은 방식
# 동일: np.expand_dims(x, -1)
print(x_train_full.shape, x_train_full.dtype, x_train_full.min(), x_train_full.max())   # (60000, 28, 28, 1) float32 0.0 1.0
print('class names:', CLASS_NAMES)

### E02 정답 — 훈련/검증 데이터 분리 (stratify) — test는 손대지 않기
**Expected Output:** train (48000, 28, 28, 1), val (12000, 28, 28, 1), test (10000, 28, 28, 1); 클래스별 4800 / 1200

**Modification Point:** 비율 변경은 `test_size`. 증강(generator)을 쓰면 `validation_split` 사용 불가 → 반드시 이렇게 명시적 분리.

In [ ]:
# ===== E02 정답 =====
from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()
x_train_full = x_train_full.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)   # 원래 train에서만 분리
print('train', x_train.shape, 'val', x_val.shape, 'test', x_test.shape)
print('train per class:', np.bincount(y_train))      # 각 4800
print('val   per class:', np.bincount(y_val))        # 각 1200
# 슬라이싱 방식: x_train, x_val = x_train_full[:48000], x_train_full[48000:]  (섞이지 않음, 이 데이터는 이미 섞여 있음)
# 같은 결과: load_fashion() ← 0번 SETUP

### E03 정답 — 훈련 이미지 10개 2×5 그리드 시각화 (클래스 이름 제목)
**Expected Output:** 2행 5열 이미지 그리드 2개 (제목: 클래스 번호/이름)

**Modification Point:** 개수가 바뀌면 `plt.subplots(r, c)`와 `figsize`. 원본(uint8, 정규화 전)도 같은 코드로 표시 가능.

In [ ]:
# ===== E03 정답 =====
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):                     # 2×5 = 10칸을 1차원으로 순회
    ax.imshow(x_train[i].squeeze(), cmap='gray')       # (28,28,1) → (28,28)
    ax.set_title(f'{y_train[i]}: {CLASS_NAMES[y_train[i]]}', fontsize=9)
    ax.axis('off')
plt.suptitle('First 10 training images'); plt.tight_layout(); plt.show()
# 변형: 클래스별 1장씩
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c, ax in enumerate(axes.flat):
    idx = np.where(y_train == c)[0][0]
    ax.imshow(x_train[idx, :, :, 0], cmap='gray'); ax.set_title(CLASS_NAMES[c], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

### E04 정답 — 훈련 데이터에만 실시간 데이터 증강 2가지 이상 (ImageDataGenerator.flow)
**Expected Output:** batch: (64, 28, 28, 1) (64,) range: 0.0 ~ 1.0, steps/epoch: 750

**Modification Point:** 증강 종류는 인자만 추가/삭제. 강의 방식(Lecture10 p.29)은 "데이터셋 변경 없이 실시간 증강" = `fit(datagen.flow(...))`. 대체: E12 Keras Random* 레이어.

In [ ]:
# ===== E04 정답 =====
from tensorflow.keras.preprocessing.image import ImageDataGenerator
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()       # 이미 0~1 정규화
train_datagen = ImageDataGenerator(
    rotation_range=10,          # ① 회전 ±10°
    width_shift_range=0.1,      # ② 좌우 이동 10%
    height_shift_range=0.1,     #    상하 이동 10%
    zoom_range=0.1,             # ③ 확대/축소
    horizontal_flip=True        # ④ 좌우 반전 (옷은 OK, 숫자 MNIST는 False — Lecture10 p.29)
)                               # rescale 넣지 않음: 이미 /255 했으므로 이중 정규화 방지
train_gen = train_datagen.flow(x_train, y_train, batch_size=64, shuffle=True, seed=42)
xb, yb = next(train_gen)
print('batch:', xb.shape, yb.shape, 'range:', xb.min().round(3), xb.max().round(3))
print('steps/epoch:', len(train_gen))                                 # ceil(48000/64) = 750
# 학습 시 (모델은 E05): 검증은 증강 없이 원본 그대로
# history = model.fit(train_gen, epochs=EPOCHS, validation_data=(x_val, y_val))
# test_loss, test_acc = model.evaluate(x_test, y_test)
# 이유: 검증/테스트는 "실제로 들어올 데이터"에 대한 성능을 재야 하므로 원본 분포 유지. 증강은 학습 다양성용.
# 주의: generator 입력에는 validation_split 사용 불가 → E02처럼 명시적으로 분리

### E05 정답 — Conv2D 기반 CNN 설계·학습·평가 + 파라미터 수
**Expected Output:** Total params 421,642. 10 epoch 기준 test accuracy ≈ 0.90 내외(증강 때문에 train acc가 val보다 낮을 수 있음)

**Modification Point:** padding 없음(valid)이면 크기 28→26→13→11→5 → Flatten 5*5*64. 라벨 원-핫이면 `categorical_crossentropy`. 클래스 수 = Dense 마지막 units.

In [ ]:
# ===== E05 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
tf.keras.utils.set_random_seed(42)
EPOCHS = 10                 # 시간 부족 시 3~5 (GPU 런타임 권장: 런타임 > 런타임 유형 변경 > GPU)
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
train_gen = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1,
                               zoom_range=0.1, horizontal_flip=True).flow(x_train, y_train, batch_size=64, seed=42)
cnn_model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),   # 3*3*1*32 + 32 = 320
    MaxPooling2D((2, 2)),                                     # 28 → 14
    Conv2D(64, (3, 3), activation='relu', padding='same'),   # 3*3*32*64 + 64 = 18,496
    MaxPooling2D((2, 2)),                                     # 14 → 7
    Flatten(),                                                # 7*7*64 = 3136
    Dense(128, activation='relu'),                            # 3136*128 + 128 = 401,536
    Dense(10, activation='softmax')                           # 128*10 + 10 = 1,290
])
cnn_model.summary()                                           # Total 421,642
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_history = cnn_model.fit(train_gen, epochs=EPOCHS, validation_data=(x_val, y_val))
test_loss, test_acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

### E06 정답 — Weight decay 적용: (A) kernel_regularizer=l2 (B) AdamW
**Expected Output:** regularization terms: 3. val accuracy 3 epoch 값 2줄 (L2 모델의 loss 값은 penalty 포함이라 더 큼)

**Modification Point:** Keras `l2(0.001)`은 λ‖W‖²에서 λ=0.001 (강의 표기 λ/2=1e-3, Lecture10 p.22). bias 규제는 `bias_regularizer`(보통 안 함).

In [ ]:
# ===== E06 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW
tf.keras.utils.set_random_seed(42)
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
def build_cnn(reg=None):
    return Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg), MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg), MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu', kernel_regularizer=reg),
        Dense(10, activation='softmax')])
# (A) Adam + layer-wise L2: L_total = L_data + Σ λ‖W‖²
model_l2 = build_cnn(reg=l2(1e-4))
model_l2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('regularization terms:', len(model_l2.losses))          # 3 (Conv, Conv, Dense)
# (B) AdamW: 손실에 penalty 없이 업데이트 때 W ← (1-αλ)W 를 분리 적용
model_wd = build_cnn()
model_wd.compile(optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h_l2 = model_l2.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=3, batch_size=128, verbose=0)
h_wd = model_wd.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=3, batch_size=128, verbose=0)
print('val acc  L2:', np.round(h_l2.history['val_accuracy'], 4), '| AdamW:', np.round(h_wd.history['val_accuracy'], 4))
# 차이: L2(A)는 penalty 기울기가 Adam의 적응형 스케일에 섞임 → 암묵적 shrink (Lecture10 p.20)
#       AdamW(B)는 weight decay가 gradient 업데이트와 분리(decoupled) (Lecture10 p.21, Lecture8 p.13)

### E07 정답 — 과적합 방지: Dropout + EarlyStopping(restore_best_weights) + ModelCheckpoint
**Expected Output:** EarlyStopping으로 30 epoch 전에 종료, best epoch의 train/val gap 출력

**Modification Point:** `patience` 크기 조절. 분류에서 정확도 기준이면 `monitor="val_accuracy", mode="max"`.

In [ ]:
# ===== E07 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
tf.keras.utils.set_random_seed(42)
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'), MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'), MaxPooling2D((2, 2)),
    Flatten(), Dropout(0.3),                                  # 학습 때만 동작, 추론 때 자동 비활성 (Lecture10 p.24)
    Dense(128, activation='relu'), Dropout(0.5),
    Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
os.makedirs('models', exist_ok=True)
callbacks = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
             ModelCheckpoint('models/best_cnn.keras', monitor='val_loss', save_best_only=True)]
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=30, batch_size=128,
                    callbacks=callbacks, verbose=2)
h = pd.DataFrame(history.history); best = h['val_loss'].idxmin()
print(f"best epoch {best + 1}: train acc {h['accuracy'][best]:.4f} / val acc {h['val_accuracy'][best]:.4f} / gap {h['accuracy'][best] - h['val_accuracy'][best]:.4f}")
print('test acc:', round(load_model('models/best_cnn.keras').evaluate(x_test, y_test, verbose=0)[1], 4))
# 과적합 방지 기법: ① Data augmentation(E04) ② Weight decay/L2(E06) ③ Dropout ④ Early stopping
#                ⑤ (모델 용량 축소, BatchNorm은 안정화 목적 — Lecture10 p.6~7)

### E08 정답 — Confusion matrix 출력 (ConfusionMatrixDisplay, 정규화, 가장 헷갈린 쌍)
**Expected Output:** 10×10 행렬, 보통 Shirt↔T-shirt/top, Pullover↔Coat 혼동이 가장 큼. 대각선 합/10000 = accuracy

**Modification Point:** seaborn 버전: `sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)`. 이진 분류는 `(prob >= 0.5).astype(int)` 후 동일.

In [ ]:
# ===== E08 정답 =====
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
if 'cnn_model' in globals():                       # E05를 실행했으면 그 모델 사용
    model = cnn_model; _, _, x_test, _, _, y_test = load_fashion()
else:
    model, _, (x_test, y_test) = train_cnn_quick(epochs=3)
y_prob = model.predict(x_test, verbose=0)          # (10000, 10) 확률
y_pred = y_prob.argmax(axis=1)                     # (10000,) 클래스
cm = confusion_matrix(y_test, y_pred)              # 행=실제, 열=예측
print(cm)
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('Confusion Matrix (test)'); plt.tight_layout(); plt.show()
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')      # 행 합 = 1 (클래스별 recall)
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', values_format='.2f', xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.show()
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f'가장 많이 헷갈린 쌍: 실제 {CLASS_NAMES[i]} → 예측 {CLASS_NAMES[j]} ({off[i, j]}개)')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

### E09 정답 — [오류 수정] CNN 파이프라인의 7가지 실수
**Expected Output:** ①test 미정규화 ②rescale 이중 정규화 ③채널(28,28,1) 누락 ④출력 relu→softmax ⑤정수 라벨에 categorical ⑥generator+validation_split 불가(ValueError) ⑦test 증강 금지

**Modification Point:** 원-핫 라벨을 쓰려면 ⑤를 그대로 두고 `to_categorical(y, 10)`.

In [ ]:
# ===== E09 정답 =====
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import train_test_split
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0         # ③ 채널 차원 추가
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0           # ① test도 같은 정규화
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2,
                                                  random_state=42, stratify=y_train)   # ⑥ 명시적 검증 분리
datagen = ImageDataGenerator(rotation_range=10, zoom_range=0.1)            # ② rescale 제거 (이중 정규화 방지)
model = Sequential([Input(shape=(28, 28, 1)),                              # ③
                    Conv2D(32, (3, 3), activation='relu'), MaxPooling2D((2, 2)),
                    Flatten(), Dense(64, activation='relu'),
                    Dense(10, activation='softmax')])                      # ④ 다중분류 출력 = softmax
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])   # ⑤ 정수 라벨
model.fit(datagen.flow(x_train, y_train, batch_size=64), validation_data=(x_val, y_val), epochs=5)   # ⑥ generator는 validation_split 불가
print(model.evaluate(x_test, y_test, verbose=0))                           # ⑦ test는 증강 없이 원본

### E10 정답 — CNN에 BatchNormalization 추가 (Conv → BN → ReLU)
**Expected Output:** BN 모델이 초기 epoch에서 val accuracy가 더 빨리 오르는 경향 (항상 그런 것은 아님; Lab2 p.17: 작은 표 데이터에선 BN이 악화)

**Modification Point:** Dense 모델(Lecture9 p.30)은 `Dense(256) → BatchNormalization() → Activation("relu")` 동일 패턴.

In [ ]:
# ===== E10 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, BatchNormalization, Activation
tf.keras.utils.set_random_seed(42)
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
def conv_bn(filters):
    return [Conv2D(filters, (3, 3), padding='same', use_bias=False),   # BN의 β가 bias 역할
            BatchNormalization(), Activation('relu')]
model_bn = Sequential([Input(shape=(28, 28, 1)),
                       *conv_bn(32), MaxPooling2D((2, 2)),
                       *conv_bn(64), MaxPooling2D((2, 2)),
                       Flatten(), Dense(128, use_bias=False), BatchNormalization(), Activation('relu'),
                       Dense(10, activation='softmax')])
model_plain = Sequential([Input(shape=(28, 28, 1)),
                          Conv2D(32, (3, 3), activation='relu', padding='same'), MaxPooling2D((2, 2)),
                          Conv2D(64, (3, 3), activation='relu', padding='same'), MaxPooling2D((2, 2)),
                          Flatten(), Dense(128, activation='relu'), Dense(10, activation='softmax')])
for name, m in [('plain', model_plain), ('BN', model_bn)]:
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=3, batch_size=128, verbose=0)
    print(name, 'val acc per epoch:', np.round(h.history['val_accuracy'], 4))
# BN: 학습 때는 mini-batch 평균/분산으로 정규화 + running 통계(EWMA) 갱신,
#     추론(evaluate/predict) 때는 저장된 running mean/var 사용 → test 통계 계산 X (Lecture9 p.29)

### E11 정답 — 증강 결과 시각화: 원본 1장 + 증강 9장 (2×5)
**Expected Output:** 원본 1장 + 서로 다른 증강 9장, aug range ≈ 0 ~ 1

**Modification Point:** 증강 강도를 바꿔가며 눈으로 확인 후 학습에 적용. 너무 강하면(예: rotation 90) 클래스 특징 훼손.

In [ ]:
# ===== E11 정답 =====
from tensorflow.keras.preprocessing.image import ImageDataGenerator
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
datagen = ImageDataGenerator(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
                             zoom_range=0.1, horizontal_flip=True)
img = x_train[0:1]                                   # (1, 28, 28, 1) — 배치 차원 유지
it = datagen.flow(img, batch_size=1, seed=1)         # y 없이 x만 → 배치만 반환
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes.flat[0].imshow(img[0, :, :, 0], cmap='gray'); axes.flat[0].set_title('original')
vals = []
for k, ax in enumerate(axes.flat[1:], start=1):
    aug = next(it)[0, :, :, 0]; vals.append(aug)
    ax.imshow(aug, cmap='gray'); ax.set_title(f'aug {k}')
for ax in axes.flat: ax.axis('off')
plt.suptitle(CLASS_NAMES[y_train[0]]); plt.tight_layout(); plt.show()
print('aug range:', np.min(vals).round(3), np.max(vals).round(3))
# 한 장만 변환: datagen.random_transform(x_train[0])

### E12 정답 — [대체 구현] Keras 전처리 레이어로 모델 내부 증강 (RandomFlip/Rotation/Translation/Zoom)
**Expected Output:** inference 동일? True / training 동일? False, 3 epoch 학습 로그와 test acc

**Modification Point:** 이 방식은 일반 배열 입력이므로 `validation_split` 사용 가능. 회전 인자는 "비율×360°"(ImageDataGenerator는 도 단위)임에 주의.

In [ ]:
# ===== E12 정답 =====
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
tf.keras.utils.set_random_seed(42)
x_train, x_val, x_test, y_train, y_val, y_test = load_fashion()
data_augmentation = Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),            # ±0.05 × 360° ≈ ±18°
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1)], name='augmentation')
sample = x_val[:8]
print('inference 동일?', np.allclose(data_augmentation(sample, training=False).numpy(), sample))   # True
print('training  동일?', np.allclose(data_augmentation(sample, training=True).numpy(), sample))    # False
model = Sequential([
    layers.Input(shape=(28, 28, 1)),
    data_augmentation,                      # fit 때만 증강, evaluate/predict 때 통과
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'), layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'), layers.MaxPooling2D((2, 2)),
    layers.Flatten(), layers.Dropout(0.3), layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=3, batch_size=128, verbose=2)
print('test acc:', round(model.evaluate(x_test, y_test, verbose=0)[1], 4))

# F. 최적화 · 안정화 · 초기화 (Lecture 8·9·10)
- **F01** [중] Optimizer 비교: SGD / Momentum / Nesterov / RMSprop / Adam / AdamW
- **F02** [상] 학습률 스케줄: Warm-up + Cosine decay (LearningRateScheduler) + ReduceLROnPlateau
- **F03** [중] 가중치 초기화 실험: 너무 작은/큰 분산 vs Xavier vs He (NumPy) + Keras initializer
- **F04** [하] 출력층·손실·은닉 활성화 짝 맞추기 — 빈칸 채우기
- **F05** [최상] NumPy로 GD vs Mini-batch SGD + L2 정규화 직접 구현 (슬라이드 수치 재현)

## F — 문제

### F01. Optimizer 비교: SGD / Momentum / Nesterov / RMSprop / Adam / AdamW
**유형:** 최적화 / 모델 수정 + 시각화 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** Fashion-MNIST 일부(1만 장, 784 벡터)로 같은 MLP를 6가지 optimizer로 5 epoch씩 학습하고 val_loss 곡선을 한 그래프에 비교하라.

**Search Keywords:** `SGD(learning_rate=, momentum=0.9)` · `nesterov=True` · `RMSprop` · `Adam` · `AdamW` · `tensorflow.keras.optimizers`

**Source:** Lecture8 p.5~13 (GD/SGD, Momentum, NAG, RMSProp, Adam, AdamW), p.17 (Adam 1e-3)

In [ ]:
# F01 연습 셀 — 빈칸(____)/TODO를 채워 실행
from tensorflow.keras.optimizers import SGD, RMSprop, Adam, AdamW
# TODO

### F02. 학습률 스케줄: Warm-up + Cosine decay (LearningRateScheduler) + ReduceLROnPlateau
**유형:** 최적화 / 코드 작성 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** (1) Lecture8 p.20의 warm-up+cosine 함수를 **epoch 단위**로 바꿔 `lr_schedule(epoch, lr)` 구현하고 20 epoch 값을 그래프로 확인
(2) `LearningRateScheduler`로 학습에 적용 (3) Lab2 방식 `ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)` 적용
(4) history에 기록된 learning rate 확인.

**Search Keywords:** `LearningRateScheduler` · `ReduceLROnPlateau` · `math.cos` · `warmup` · `factor` · `min_lr` · `history.history['learning_rate']`

**Source:** Lecture8 p.17~20 (lr 선택, warm-up, cosine 코드), Lecture11_Lab2 p.22 (ReduceLROnPlateau)

In [ ]:
# F02 연습 셀 — 빈칸(____)/TODO를 채워 실행
import math
from tensorflow.keras.callbacks import LearningRateScheduler, ReduceLROnPlateau
base_lr, warmup_epochs, total_epochs = 1e-3, 3, 20
def lr_schedule(epoch, lr=None):
    ____

### F03. 가중치 초기화 실험: 너무 작은/큰 분산 vs Xavier vs He (NumPy) + Keras initializer
**유형:** 최적화(초기화) / 시각화 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 중

**문제:** (1) NumPy로 6층(각 4096 유닛, Lecture8 p.24~25 설정) tanh 신경망에 랜덤 입력을 통과시키며 층별 활성화 표준편차를 비교: σ=0.01, σ=0.05, Xavier(√(1/n_in))
(2) ReLU에서 Xavier vs He(√(2/n_in)) 비교 (3) Keras에서 `kernel_initializer` 지정법과 실제 가중치 표준편차 확인.

**Search Keywords:** `np.random.randn` · `np.sqrt(2 / n_in)` · `glorot_normal` · `he_normal` · `lecun_normal` · `kernel_initializer` · `activation histogram`

**Source:** Lecture8 p.22~29 (초기화, 너무 작은/큰 분산, LeCun/Xavier/He), Lecture11_Lab2 p.14 (he_normal)

In [ ]:
# F03 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
def forward_stats(init_std_fn, act, n=4096, layers=6):
    ____

### F04. 출력층·손실·은닉 활성화 짝 맞추기 — 빈칸 채우기
**유형:** 모델 구성 / 빈칸 채우기 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 하

**문제:** 각 과제에 맞게 출력층 활성화·유닛 수·손실의 빈칸을 채우고, 은닉층에 `LeakyReLU(α=0.1)`를 쓰는 코드를 작성하라.

**Search Keywords:** `activation='linear'` · `sigmoid` · `softmax` · `mse` · `binary_crossentropy` · `categorical_crossentropy` · `LeakyReLU(negative_slope=0.1)` · `PReLU` · `activation='gelu'`

**Source:** Lecture9 p.12~13 (활성화 함수 Keras 코드, 출력층), Lecture6_Lab1 p.28~29

In [ ]:
# F04 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 회귀(출력 2개):    Dense(____, activation='____'),  loss='____'
# 이진분류:          Dense(____, activation='____'),  loss='____'
# 다중분류(10, 정수): Dense(____, activation='____'),  loss='____'
# 다중라벨(5):       Dense(____, activation='____'),  loss='____'

### F05. NumPy로 GD vs Mini-batch SGD + L2 정규화 직접 구현 (슬라이드 수치 재현)
**유형:** 최적화 / 직접 구현 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 최상

**문제:** (1) Lecture8 p.7 데이터(공부시간 1,3,5,7,9,11 → 점수 75,77,85,83,90,91)에 `ŷ = w1 + w2·x`를 **Full-batch GD**와 **Mini-batch SGD(B=2)**로 학습해 `np.polyfit` 결과와 비교, epoch당 업데이트 횟수 확인
(2) Lecture10 p.15 데이터(1,3,5,7 → 75,77,85,83)에 `L = mean((wx+b−y)²) + λw²`를 GD로 최소화해 λ=0 → (1.6, 73.6), λ=1 → (1.33, 74.67) 재현.

**Search Keywords:** `gradient descent` · `np.mean` · `mini-batch` · `np.random.permutation` · `learning rate` · `np.polyfit` · `L2 penalty` · `lambda`

**Source:** Lecture8 p.5~8 (GD vs SGD, 6개 데이터, 3 updates/epoch), Lecture10 p.15 (L2 예제 수치), Lecture5 p.16

In [ ]:
# F05 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np
x = np.array([1, 3, 5, 7, 9, 11.]); y = np.array([75, 77, 85, 83, 90, 91.])
# TODO

## F — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### F01 정답 — Optimizer 비교: SGD / Momentum / Nesterov / RMSprop / Adam / AdamW
**Expected Output:** SGD(무모멘텀)가 가장 느리고 Momentum/Adam 계열이 빠르게 감소하는 경향

**Modification Point:** optimizer 객체는 재사용하지 말고 매번 새로 생성. 문자열 `"adam"`은 기본 lr=1e-3.

In [ ]:
# ===== F01 정답 =====
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import SGD, RMSprop, Adam, AdamW
from tensorflow.keras.datasets import fashion_mnist
(x, y), _ = fashion_mnist.load_data()
x = x[:12000].reshape(-1, 784).astype('float32') / 255.0; y = y[:12000]
x_tr, y_tr, x_va, y_va = x[:10000], y[:10000], x[10000:], y[10000:]
opts = {'SGD': lambda: SGD(learning_rate=0.01),
        'Momentum': lambda: SGD(learning_rate=0.01, momentum=0.9),
        'Nesterov': lambda: SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
        'RMSprop': lambda: RMSprop(learning_rate=1e-3),
        'Adam': lambda: Adam(learning_rate=1e-3),
        'AdamW': lambda: AdamW(learning_rate=1e-3, weight_decay=1e-4)}
for name, make_opt in opts.items():
    tf.keras.utils.set_random_seed(42)                       # 같은 초기값에서 비교
    m = Sequential([Input(shape=(784,)), Dense(128, activation='relu'), Dense(10, activation='softmax')])
    m.compile(optimizer=make_opt(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])   # optimizer는 모델마다 새로
    h = m.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=5, batch_size=64, verbose=0)
    plt.plot(range(1, 6), h.history['val_loss'], 'o-', label=name)
    print(f"{name:9s} val_acc={h.history['val_accuracy'][-1]:.4f}")
plt.xlabel('epoch'); plt.ylabel('val_loss'); plt.legend(); plt.grid(True); plt.show()

### F02 정답 — 학습률 스케줄: Warm-up + Cosine decay (LearningRateScheduler) + ReduceLROnPlateau
**Expected Output:** lr: 3.3e-4, 6.7e-4, 1e-3 → cosine으로 0 근처까지 감소. ReduceLROnPlateau는 val_loss 정체 시 절반씩

**Modification Point:** Lab2 설정: `monitor="val_mae", factor=0.5, patience=15, cooldown=5, min_lr=1e-5, mode="min"`.

In [ ]:
# ===== F02 정답 =====
import math, tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.datasets import fashion_mnist
base_lr, warmup_epochs, total_epochs = 1e-3, 3, 20
def lr_schedule(epoch, lr=None):
    if epoch < warmup_epochs:                                  # 1. warm-up: 작게 시작 → base_lr
        return float(base_lr * (epoch + 1) / warmup_epochs)
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return float(base_lr * 0.5 * (1 + math.cos(math.pi * progress)))   # 2. cosine: base_lr → 0
plt.plot([lr_schedule(e) for e in range(total_epochs)], 'o-'); plt.xlabel('epoch'); plt.ylabel('lr'); plt.show()
(x, y), _ = fashion_mnist.load_data()
x = x[:12000].reshape(-1, 784).astype('float32') / 255.0; y = y[:12000]
def build():
    tf.keras.utils.set_random_seed(42)
    m = Sequential([Input(shape=(784,)), Dense(128, activation='relu'), Dense(10, activation='softmax')])
    m.compile(optimizer=tf.keras.optimizers.Adam(base_lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m
h1 = build().fit(x, y, validation_split=0.2, epochs=total_epochs, batch_size=64, verbose=0,
                 callbacks=[LearningRateScheduler(lr_schedule)])
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)
h2 = build().fit(x, y, validation_split=0.2, epochs=total_epochs, batch_size=64, verbose=0, callbacks=[reduce_lr])
for name, h in [('warmup+cosine', h1), ('ReduceLROnPlateau', h2)]:
    lrs = h.history.get('learning_rate', h.history.get('lr', []))   # Keras 3: 'learning_rate'
    print(name, np.round(lrs, 6))
# step 단위 내장: tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate, decay_steps, warmup_target=, warmup_steps=)

### F03 정답 — 가중치 초기화 실험: 너무 작은/큰 분산 vs Xavier vs He (NumPy) + Keras initializer
**Expected Output:** 
```text
tanh  σ=0.01  : [0.492, 0.289, 0.179, 0.113, 0.072, 0.046]
tanh  σ=0.05  : [0.872, 0.853, 0.85, 0.849, 0.849, 0.849]
tanh  Xavier  : [0.627, 0.486, 0.408, 0.357, 0.321, 0.294]
ReLU  Xavier  : [0.584, 0.412, 0.295, 0.207, 0.147, 0.104]
ReLU  He      : [0.825, 0.827, 0.825, 0.82, 0.829, 0.828]
```

**Modification Point:** He는 ReLU·LeakyReLU·ELU, Xavier(glorot)는 sigmoid·tanh (Lecture8 p.28~29). n=4096·입력 N(0,1)이면 슬라이드 히스토그램의 std와 거의 같다.

In [ ]:
# ===== F03 정답 =====
import numpy as np
np.random.seed(0)
def forward_stats(init_std_fn, act, n=4096, layers=6):
    h = np.random.randn(500, n); stds = []
    for _ in range(layers):
        W = np.random.randn(n, n) * init_std_fn(n)          # W ~ N(0, σ²)
        h = act(h @ W); stds.append(round(float(h.std()), 3))
    return stds
tanh, relu = np.tanh, lambda z: np.maximum(0, z)
print('tanh  σ=0.01  :', forward_stats(lambda n: 0.01, tanh))               # 점점 0으로 → vanishing
print('tanh  σ=0.05  :', forward_stats(lambda n: 0.05, tanh))               # ±1 포화
print('tanh  Xavier  :', forward_stats(lambda n: np.sqrt(1 / n), tanh))     # 안정
print('ReLU  Xavier  :', forward_stats(lambda n: np.sqrt(1 / n), relu))     # 점점 줄어듦
print('ReLU  He      :', forward_stats(lambda n: np.sqrt(2 / n), relu))     # 안정
try:                                                                       # Keras (Colab)
    from tensorflow.keras.layers import Dense, Input
    from tensorflow.keras.models import Sequential
    m = Sequential([Input(shape=(4096,)),
                    Dense(256, activation='relu', kernel_initializer='he_normal', name='he_dense'),
                    Dense(256, activation='tanh', kernel_initializer='glorot_uniform'),   # Keras 기본값
                    Dense(10, activation='softmax')])
    W1 = m.get_layer('he_dense').get_weights()[0]
    print('he_normal std:', W1.std().round(4), '| 이론 sqrt(2/4096) =', np.sqrt(2 / 4096).round(4), '(truncated normal이라 약간 작음)')
except ImportError:
    print('(TensorFlow 없음: Keras 부분 생략)')

### F04 정답 — 출력층·손실·은닉 활성화 짝 맞추기 — 빈칸 채우기
**Expected Output:** regression (None, 2) mse / binary (None, 1) BCE / multiclass (None, 10) SCCE / multilabel (None, 5) BCE

**Modification Point:** 원-핫 라벨이면 multiclass 손실 `categorical_crossentropy`. 구버전 Keras(2.x)는 `LeakyReLU(alpha=0.1)`.

In [ ]:
# ===== F04 정답 =====
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LeakyReLU, PReLU
heads = {'regression(2)':  (Dense(2, activation='linear'), 'mse'),
         'binary':         (Dense(1, activation='sigmoid'), 'binary_crossentropy'),
         'multiclass(10)': (Dense(10, activation='softmax'), 'sparse_categorical_crossentropy'),
         'multilabel(5)':  (Dense(5, activation='sigmoid'), 'binary_crossentropy')}
for task, (out, loss) in heads.items():
    m = Sequential([Input(shape=(8,)),
                    Dense(32), LeakyReLU(negative_slope=0.1),     # Keras 3: negative_slope (구버전 alpha)
                    Dense(16), PReLU(),                           # α 학습
                    out])
    m.compile(optimizer='adam', loss=loss)
    print(f'{task:15s} output shape {tuple(m.output.shape)} loss={loss}')
# 기타: Dense(32, activation='elu' | 'gelu' | 'swish')  (Lecture9 p.13)

### F05 정답 — NumPy로 GD vs Mini-batch SGD + L2 정규화 직접 구현 (슬라이드 수치 재현)
**Expected Output:** 
```text
GD       w1(bias)=73.471 w2(slope)=1.671
SGD      w1=73.519 w2=1.624  (updates/epoch = 3, 노이즈로 근처에서 진동)
polyfit  slope, bias = [ 1.671 73.471]
λ=0.0: w=1.600, b=73.600
λ=1.0: w=1.333, b=74.667
```

**Modification Point:** lr이 너무 크면 발산(x 범위가 클수록 위험) → 입력 표준화하면 큰 lr로도 빠르게 수렴 (Lecture9 p.15~17). 배치 크기 B를 바꾸면 updates/epoch = ceil(N/B).

In [ ]:
# ===== F05 정답 =====
import numpy as np
x = np.array([1, 3, 5, 7, 9, 11.]); y = np.array([75, 77, 85, 83, 90, 91.])
def grads(w1, w2, xb, yb):                     # L = mean((w1 + w2 x - y)^2)
    e = w1 + w2 * xb - yb
    return 2 * e.mean(), 2 * (e * xb).mean()
# (1-a) Full-batch GD: epoch당 1회 업데이트
w1 = w2 = 0.0; lr = 0.01
for epoch in range(20000):
    g1, g2 = grads(w1, w2, x, y); w1 -= lr * g1; w2 -= lr * g2
print('GD       w1(bias)=%.3f w2(slope)=%.3f' % (w1, w2))
# (1-b) Mini-batch SGD (B=2): epoch당 3회 업데이트, 비복원 추출
rng = np.random.default_rng(0); v1 = v2 = 0.0; updates = 0
for epoch in range(20000):
    idx = rng.permutation(len(x))
    for b in range(0, len(x), 2):
        bi = idx[b:b + 2]; g1, g2 = grads(v1, v2, x[bi], y[bi])
        v1 -= lr * g1; v2 -= lr * g2; updates += 1
print('SGD      w1=%.3f w2=%.3f  (updates/epoch = %d, 노이즈로 근처에서 진동)' % (v1, v2, updates // 20000))
print('polyfit  slope, bias =', np.round(np.polyfit(x, y, 1), 3))
# (2) L2 regularization: L = mean((w x + b - y)^2) + λ w^2  (bias는 규제 X)
xs = np.array([1, 3, 5, 7.]); ys = np.array([75, 77, 85, 83.])
for lam in [0.0, 1.0]:
    w = b = 0.0
    for _ in range(50000):
        e = w * xs + b - ys
        w -= 0.01 * (2 * (e * xs).mean() + 2 * lam * w)
        b -= 0.01 * (2 * e.mean())
    print(f'λ={lam}: w={w:.3f}, b={b:.3f}')      # λ↑ → slope w 감소

# G. 통합 실습 (시험형 대문항)
- **G01** [최상] [통합] Fashion-MNIST CNN 전체 파이프라인 (족보 항목 9개 연결)
- **G02** [최상] [통합] MSFT 2차원 출력 회귀 (결측치 → 라벨링·슬라이싱 → 사용자 정의 손실 → 시각화)
- **G03** [상] [통합] NumPy 게임 기록 → Pandas DataFrame 라벨링·슬라이싱·결측치·시각화
- **G04** [상] [통합] Pima 이진 분류: 6:2:2 분할 → 표준화 → Dropout·L2 MLP → EarlyStopping → 평가지표·혼동행렬

## G — 문제

### G01. [통합] Fashion-MNIST CNN 전체 파이프라인 (족보 항목 9개 연결)
**유형:** 통합 실습 / CNN · **출제 근거:** 족보 직접 등장 · **난이도:** 최상

**문제:** 다음 소문항을 순서대로 모두 수행하라.
1. Fashion-MNIST를 불러와 0~1 정규화하고 `(N,28,28,1)`로 변환
2. 훈련 데이터의 20%를 검증 데이터로 분리 (stratify)
3. 훈련 이미지 10개를 2×5 그리드로 시각화 (클래스 이름)
4. **훈련 데이터에만** 실시간 데이터 증강 2가지 이상 적용
5. Conv2D 기반 CNN 설계 (Conv 2블록 이상)
6. Weight decay 적용
7. 과적합 방지 기법(Dropout + EarlyStopping) 적용 후 학습
8. 학습곡선(loss, accuracy) 시각화
9. 테스트 정확도와 Confusion matrix 출력

**Search Keywords:** `fashion_mnist` · `/255.0` · `train_test_split stratify` · `plt.subplots(2, 5)` · `ImageDataGenerator` · `Conv2D` · `kernel_regularizer=l2` · `AdamW` · `Dropout` · `EarlyStopping` · `history.history` · `confusion_matrix` · `ConfusionMatrixDisplay`

**Source:** 족보 Fashion-MNIST 문항 전체; Lecture10 p.22, 26, 29; Lecture4 p.24; Lecture6_Lab1 p.39~42

In [ ]:
# G01 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 각 STEP 셀에 작성. 막히면 E01~E08 정답 셀의 코드를 Ctrl+F로 찾아 붙여넣기

### G02. [통합] MSFT 2차원 출력 회귀 (결측치 → 라벨링·슬라이싱 → 사용자 정의 손실 → 시각화)
**유형:** 통합 실습 / 회귀 · **출제 근거:** 족보 직접 등장 · **난이도:** 최상

**문제:** MSFT 주가 데이터로 다음 소문항을 수행하라.
1. 데이터를 불러와 결측치 개수를 확인하고 제거
2. `Date`를 인덱스로 두고, 다음날 시가·종가를 라벨(`Next_Open`, `Next_Close`)로 생성
3. `loc`/`iloc`으로 입력 X(5개 열)와 출력 y(2개 열)를 슬라이싱하고, 시간순으로 train/val/test 분할
4. 입력·출력 표준화 (train 기준)
5. **사용자 정의 손실함수**(가중 MSE 또는 Huber) 정의
6. 출력 2차원 MLP 구성·학습 (EarlyStopping)
7. 학습곡선 시각화 8. test 실제값 vs 예측값 그래프 및 출력별 MAE

**Search Keywords:** `dropna` · `shift(-1)` · `set_index` · `loc` · `iloc` · `StandardScaler` · `custom loss function` · `Dense(2)` · `EarlyStopping` · `inverse_transform` · `plt.subplots(2, 1)`

**Source:** 족보 MSFT 회귀 문항 전체; Lecture9 p.20; Lecture10 p.16; Lecture11_Lab2 p.46~49

In [ ]:
# G02 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 각 STEP 셀에 작성. 막히면 C01~C05 정답 참고

### G03. [통합] NumPy 게임 기록 → Pandas DataFrame 라벨링·슬라이싱·결측치·시각화
**유형:** 통합 실습 / NumPy + Pandas · **출제 근거:** 족보 직접 등장 · **난이도:** 상

**문제:** 1. seed=2026으로 플레이어와 컴퓨터가 각각 주사위 2개를 굴려 합이 큰 쪽이 이기는 게임을 100판 진행
2. 판마다 `round, player, computer, result(Win/Draw/Lose)`를 DataFrame에 기록 (행 라벨 `R001`~)
3. 10% 확률로 컴퓨터 기권(`computer = NaN`) → 결측치 개수 확인 후 해당 판 제거
4. `diff = player − computer`, `big_win = diff ≥ 5` 라벨 열 생성
5. `loc`으로 R010~R020의 result, `iloc`으로 마지막 5판 출력, Win 판만 boolean 슬라이싱
6. 결과별 횟수(`value_counts`)와 누적 승률 그래프.

**Search Keywords:** `np.random.randint` · `np.where` · `np.select` · `pd.DataFrame` · `index=` · `isna().sum()` · `dropna` · `loc` · `iloc` · `value_counts` · `cumsum` · `expanding().mean()`

**Source:** 족보 "NumPy 난수 생성 및 조건문을 이용한 게임" + "결측치 제거" + "DataFrame 라벨링 및 슬라이싱" 결합 (일반 구현)

In [ ]:
# G03 연습 셀 — 빈칸(____)/TODO를 채워 실행
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(2026)
# TODO

### G04. [통합] Pima 이진 분류: 6:2:2 분할 → 표준화 → Dropout·L2 MLP → EarlyStopping → 평가지표·혼동행렬
**유형:** 통합 실습 / 분류 · **출제 근거:** 강의자료 기반 예상 · **난이도:** 상

**문제:** 1. Pima 데이터를 stratify로 train/val/test = 6:2:2 분할
2. StandardScaler (train 기준)
3. `Dense(12)–Dropout(0.2)–Dense(8)–Dense(1, sigmoid)`, 은닉층 L2(1e-3)
4. EarlyStopping(val_loss, patience=20, restore_best_weights=True)로 학습
5. 학습곡선 6. 확률 0.5 기준 예측 → confusion matrix, accuracy/precision/recall/F1
7. 임계값을 0.3으로 낮추면 recall/precision이 어떻게 변하는지 비교.

**Search Keywords:** `train_test_split stratify` · `StandardScaler` · `Dropout` · `kernel_regularizer=l2` · `EarlyStopping` · `predict` · `threshold` · `confusion_matrix` · `classification_report` · `recall_score`

**Source:** Lecture6_Lab1 p.13~16, p.33, p.41~42; Lecture4 p.24~27; Lecture9 p.20; Lecture10 p.22, p.26

In [ ]:
# G04 연습 셀 — 빈칸(____)/TODO를 채워 실행
# 0번 SETUP 실행 후
mount_drive(); df = load_pima()
# TODO

## G — 정답
정답 셀은 [0] SETUP만 실행되어 있으면 독립 실행됩니다.

### G01 정답 — [통합] Fashion-MNIST CNN 전체 파이프라인 (족보 항목 9개 연결)
**Expected Output:** STEP별 출력: shape/범위 → 분할 shape → 그리드 → summary → 학습 로그 → 곡선 2개 → test acc(약 0.88~0.91) + 10×10 행렬

**Modification Point:** 데이터가 MNIST면 `mnist` + `horizontal_flip=False`. CIFAR-10이면 `Input(32,32,3)`, reshape 생략. 시간 제한 시 EPOCHS↓, batch_size↑.

In [ ]:
# ===== G01 정답 (1/8) =====
# %% [STEP 1] 불러오기 + 0~1 정규화 (→ E01)
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split
tf.keras.utils.set_random_seed(42)
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()
x_train_full = x_train_full.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
print(x_train_full.shape, x_train_full.min(), x_train_full.max())

In [ ]:
# ===== G01 정답 (2/8) =====
# %% [STEP 2] 훈련/검증 분리 (→ E02)
x_train, x_val, y_train, y_val = train_test_split(x_train_full, y_train_full, test_size=0.2,
                                                  random_state=42, stratify=y_train_full)
print('train', x_train.shape, 'val', x_val.shape, 'test', x_test.shape)

In [ ]:
# ===== G01 정답 (3/8) =====
# %% [STEP 3] 2×5 그리드 (→ E03)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray'); ax.set_title(CLASS_NAMES[y_train[i]]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ===== G01 정답 (4/8) =====
# %% [STEP 4] 훈련 데이터에만 실시간 증강 (→ E04)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1,
                                   zoom_range=0.1, horizontal_flip=True)          # 4가지
train_gen = train_datagen.flow(x_train, y_train, batch_size=64, shuffle=True, seed=42)
# 검증·테스트: 증강 없이 (x_val, y_val), (x_test, y_test) 그대로 사용

In [ ]:
# ===== G01 정답 (5/8) =====
# %% [STEP 5+6+7a] Conv2D CNN + Weight decay + Dropout (→ E05, E06, E07)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2
WD = 1e-4
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(WD)), MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(WD)), MaxPooling2D((2, 2)),
    Flatten(), Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(WD)), Dropout(0.5),
    Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# 대안(6): model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4), ...)
model.summary()

In [ ]:
# ===== G01 정답 (6/8) =====
# %% [STEP 7b] EarlyStopping과 함께 학습
from tensorflow.keras.callbacks import EarlyStopping
EPOCHS = 15                      # 시간 부족 시 5
early = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(train_gen, epochs=EPOCHS, validation_data=(x_val, y_val), callbacks=[early])

In [ ]:
# ===== G01 정답 (7/8) =====
# %% [STEP 8] 학습곡선 (→ C05)
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, k in zip(axes, ['loss', 'accuracy']):
    ax.plot(hist[k], label='train ' + k); ax.plot(hist['val_' + k], label='val ' + k)
    ax.set_xlabel('epoch'); ax.set_title(k); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# ===== G01 정답 (8/8) =====
# %% [STEP 9] 테스트 평가 + Confusion matrix (→ E08)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
y_pred = model.predict(x_test, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred); print(cm)
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

### G02 정답 — [통합] MSFT 2차원 출력 회귀 (결측치 → 라벨링·슬라이싱 → 사용자 정의 손실 → 시각화)
**Expected Output:** 결측 개수 → head → 학습곡선 → 실제/예측 그래프 2개(제목에 MAE)

**Modification Point:** 손실 교체는 `model.compile(loss=...)` 한 줄. 결측치를 채우라고 하면 `df.ffill()` 또는 train 중앙값.

In [ ]:
# ===== G02 정답 (1/8) =====
# %% [STEP 1] 불러오기 + 결측치 확인·제거 (→ B02, C01)
mount_drive()
df = load_msft()                                  # 시험: pd.read_csv('주어진 경로')
print(df.isna().sum())
df = df.dropna().copy()

In [ ]:
# ===== G02 정답 (2/8) =====
# %% [STEP 2] 날짜 인덱스 + 라벨 생성 (→ B03)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')
df['Next_Open'] = df['Open'].shift(-1)
df['Next_Close'] = df['Close'].shift(-1)
df = df.dropna(subset=['Next_Open', 'Next_Close'])
print(df.loc[:, ['Open', 'Close', 'Next_Open', 'Next_Close']].head(3))

In [ ]:
# ===== G02 정답 (3/8) =====
# %% [STEP 3] 슬라이싱 + 시간순 분할 (→ B01, C01)
X = df.loc[:, ['Open', 'High', 'Low', 'Close', 'Volume']]
y = df.loc[:, ['Next_Open', 'Next_Close']]
n = len(df); i1, i2 = int(n * 0.7), int(n * 0.85)
X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

In [ ]:
# ===== G02 정답 (4/8) =====
# %% [STEP 4] 표준화 (→ B05)
from sklearn.preprocessing import StandardScaler
sx, sy = StandardScaler(), StandardScaler()
Xtr, Xva, Xte = sx.fit_transform(X_train), sx.transform(X_val), sx.transform(X_test)
ytr, yva = sy.fit_transform(y_train), sy.transform(y_val)

In [ ]:
# ===== G02 정답 (5/8) =====
# %% [STEP 5] 사용자 정의 손실 (→ C03)
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
LOSS_W = tf.constant([1.0, 2.0])                  # [Open, Close] 가중치
def weighted_mse(y_true, y_pred):
    y_true = tf.cast(y_true, y_pred.dtype)
    return tf.reduce_mean(LOSS_W * tf.square(y_true - y_pred), axis=-1)

In [ ]:
# ===== G02 정답 (6/8) =====
# %% [STEP 6] 2차원 출력 MLP (→ C02)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
model = Sequential([Input(shape=(5,)), Dense(64, activation='relu'), Dense(32, activation='relu'), Dense(2)])
model.compile(optimizer='adam', loss=weighted_mse, metrics=['mae'])
history = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=200, batch_size=32, verbose=0,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])

In [ ]:
# ===== G02 정답 (7/8) =====
# %% [STEP 7] 학습곡선 (→ C05)
plot_history(history, metric='mae')

In [ ]:
# ===== G02 정답 (8/8) =====
# %% [STEP 8] 실제 vs 예측 + MAE (→ C04)
from sklearn.metrics import mean_absolute_error
y_pred = sy.inverse_transform(model.predict(Xte, verbose=0))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for i, name in enumerate(y.columns):
    axes[i].plot(y_test.index, y_test.iloc[:, i], label='Actual')
    axes[i].plot(y_test.index, y_pred[:, i], label='Predicted')
    axes[i].set_title(f'{name}  MAE={mean_absolute_error(y_test.iloc[:, i], y_pred[:, i]):.2f}$'); axes[i].legend()
plt.tight_layout(); plt.show()

### G03 정답 — [통합] NumPy 게임 기록 → Pandas DataFrame 라벨링·슬라이싱·결측치·시각화
**Expected Output:** 
```text
결측치:
 round        0
player       0
computer    19
result       0
dtype: int64
R010    Draw
R011    Lose
R012     Win
R013     Win
R015    Lose
R016     Win
R017    Lose
R018    Lose
R019     Win
Name: result, dtype: str
      round  player  computer result  diff  big_win
R094     94      11         7    Win     4        0
R096     96       9         7    Win     2        0
R098     98       8         6    Win     2        0
R099     99       6         9   Lose    -3        0
R100    100       7        10   Lose    -3        0
Win 판 수: 37 | big_win: 6
result
Win     37
Lose    36
Draw     8
Name: count, dtype: int64
```

**Modification Point:** 규칙 변경은 `np.select`의 조건 리스트. 기권을 0점 처리하라면 `fillna(0)`.

In [ ]:
# ===== G03 정답 =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(2026)
N = 100
player = np.random.randint(1, 7, size=(N, 2)).sum(axis=1)           # 1. 벡터화 게임
computer = np.random.randint(1, 7, size=(N, 2)).sum(axis=1).astype(float)
forfeit = np.random.rand(N) < 0.10                                  # 3. 기권 → NaN
computer[forfeit] = np.nan
result = np.select([player > computer, player == computer], ['Win', 'Draw'], default='Lose')
df = pd.DataFrame({'round': np.arange(1, N + 1), 'player': player, 'computer': computer, 'result': result},
                  index=[f'R{i:03d}' for i in range(1, N + 1)])     # 2. 행 라벨
print('결측치:\n', df.isna().sum())
df = df.dropna(subset=['computer']).copy()                          # 기권 판 제거
df['computer'] = df['computer'].astype(int)
df['diff'] = df['player'] - df['computer']                          # 4. 라벨 열
df['big_win'] = np.where(df['diff'] >= 5, 1, 0)
print(df.loc['R010':'R020', 'result'])                              # 5. 라벨 슬라이싱 (끝 포함, 제거된 판은 빠짐)
print(df.iloc[-5:])
wins = df[df['result'] == 'Win']; print('Win 판 수:', len(wins), '| big_win:', df['big_win'].sum())
print(df['result'].value_counts())                                  # 6.
win_rate = (df['result'] == 'Win').expanding().mean()
plt.plot(df['round'], win_rate); plt.axhline(win_rate.iloc[-1], ls='--', c='r')
plt.xlabel('round'); plt.ylabel('cumulative win rate'); plt.show()

### G04 정답 — [통합] Pima 이진 분류: 6:2:2 분할 → 표준화 → Dropout·L2 MLP → EarlyStopping → 평가지표·혼동행렬
**Expected Output:** test accuracy ≈ 0.75~0.80. 임계값 0.3이면 recall↑, precision↓ (Lecture4 p.26 trade-off)

**Modification Point:** wine 데이터면 `load_wine_rw()`, `iloc[:, 0:12]`, `iloc[:, 12]`, `Input(shape=(12,))`. 다중분류면 E08 방식.

In [ ]:
# ===== G04 정답 (1/4) =====
# %% [STEP 1-2] 분할 + 표준화
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.keras.utils.set_random_seed(42)
mount_drive(); df = load_pima()
X = df.iloc[:, 0:8].values; y = df.iloc[:, 8].values
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
scaler = StandardScaler()
X_train, X_val, X_test = scaler.fit_transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

In [ ]:
# ===== G04 정답 (2/4) =====
# %% [STEP 3-4] 모델 + 학습
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
model = Sequential([Input(shape=(8,)),
                    Dense(12, activation='relu', kernel_regularizer=l2(1e-3)), Dropout(0.2),
                    Dense(8, activation='relu', kernel_regularizer=l2(1e-3)),
                    Dense(1, activation='sigmoid')])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=500, batch_size=16, verbose=0,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)])

In [ ]:
# ===== G04 정답 (3/4) =====
# %% [STEP 5] 학습곡선
plot_history(history, metric='accuracy')

In [ ]:
# ===== G04 정답 (4/4) =====
# %% [STEP 6-7] 평가지표 + 혼동행렬 + 임계값 비교
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, precision_score, recall_score
prob = model.predict(X_test, verbose=0).ravel()
for th in [0.5, 0.3]:
    pred = (prob >= th).astype(int)
    print(f'threshold={th}: precision={precision_score(y_test, pred):.3f} recall={recall_score(y_test, pred):.3f}')
    print(confusion_matrix(y_test, pred))
pred = (prob >= 0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['negative', 'positive']).plot(cmap='Blues'); plt.show()
print(classification_report(y_test, pred, target_names=['negative', 'positive'], digits=3))

# 최종 체크리스트 (시험 직전)
- [ ] [0] SETUP 실행, `DATA_DIR`/`MSFT_CSV` 경로를 시험 파일로 수정
- [ ] 회귀: 출력 `Dense(k)` 활성화 없음 · `loss='mse'` · y 스케일 시 `inverse_transform`
- [ ] 분류: 라벨 형식 ↔ 손실(정수→sparse, 원-핫→categorical) · 출력 softmax/sigmoid
- [ ] 스케일러/중앙값은 **train에만 fit** · 시계열은 `shuffle=False`
- [ ] 증강은 **train generator에만** · 검증은 `validation_data=(x_val, y_val)` 원본 · generator엔 `validation_split` 금지
- [ ] 이미지 `/255.0` 한 번만 (`rescale` 중복 금지) · Conv2D 입력 `(N, 28, 28, 1)`
- [ ] Weight decay: `kernel_regularizer=l2(...)` 또는 `AdamW(weight_decay=...)` · 과적합: Dropout + EarlyStopping
- [ ] Confusion matrix: `predict(...).argmax(axis=1)` → `confusion_matrix(y_true, y_pred)`